# Ranker

Ноутбук обучает ранжирующую модель поверх candidate pool из retrieval и DL-источников.

Пайплайн:

1. загружает candidate pool `merged_with_dl_all`;
2. строит point-in-time признаки без утечки будущего;
3. добавляет candidate-level признаки для retrieval-источников и DL-моделей;
4. обучает LightGBM LambdaRank;
5. опционально сравнивает с CatBoost YetiRank;
6. выбирает модель по validation `ndcg@100`;
7. скорит validation/test holdout;
8. сохраняет финальные рекомендации.

`test` используется как labeled holdout, потому что покупки за период с 2 по 15 июля 2025 доступны в datamart. Test-метрики считаются только для финальной отчетности и не используются для подбора модели.

Для ограничения потребления RAM ноутбук использует ограниченный набор признаков, downsampling fit-строк и шардовый scoring validation/test.

In [1]:
from __future__ import annotations

import gc
import json
import os
import re
import shutil
import time
import warnings
from collections import defaultdict
from dataclasses import asdict, dataclass
from datetime import date, datetime, timedelta, timezone
from pathlib import Path
from typing import Iterable, Sequence

# Ограничиваем внутренние BLAS-пулы до импорта numpy/lightgbm.
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import joblib
import lightgbm as lgbm
import numpy as np
import optuna
import pandas as pd
import polars as pl
from optuna.samplers import TPESampler
from tqdm.auto import tqdm, trange

try:
    from catboost import CatBoostRanker, Pool
    HAS_CATBOOST = True
except ImportError:
    CatBoostRanker = None
    Pool = None
    HAS_CATBOOST = False

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 200)
pl.Config.set_tbl_cols(200)
pl.Config.set_tbl_rows(30)

polars.config.Config

In [ ]:
def find_project_root(start: Path = Path.cwd()) -> Path:
    for p in [start, *start.parents]:
        if (p / "data" / "raw").exists() or (p / "data" / "datamart").exists() or (p / "data").exists():
            return p
    raise FileNotFoundError("Не найден project root с папкой data")


@dataclass(frozen=True)
class FinalRankerConfig:
    # Даты должны совпадать с datamart/candidate/DL ноутбуками.
    train_context_start: date = date(2025, 4, 18)
    train_context_end: date = date(2025, 6, 3)
    train_target_start: date = date(2025, 6, 4)
    train_target_end: date = date(2025, 6, 17)

    val_context_start: date = date(2025, 4, 18)
    val_context_end: date = date(2025, 6, 17)
    val_target_start: date = date(2025, 6, 18)
    val_target_end: date = date(2025, 7, 1)

    full_context_start: date = date(2025, 5, 2)
    full_context_end: date = date(2025, 7, 1)
    test_target_start: date = date(2025, 7, 2)
    test_target_end: date = date(2025, 7, 15)

    # В этом проекте test не blind split: покупки test target window доступны в agg/orders/user_item.
    # Поэтому test собирается с label и используется как финальный holdout, но не для подбора модели.
    test_has_labels: bool = True

    # Основной вход финального ранкера.
    candidate_input_name: str = "merged_with_dl_all"
    fallback_candidate_input_names: tuple[str, ...] = (
        "merged_with_bert4rec_small",
        "merged_with_sasrec_v3",
        "merged",
    )

    # Имена выходов. run_name не должен совпадать со старым 5_reranker, чтобы не перетереть кеши.
    run_name: str = "final_reranker_lgbm_dl"
    feature_store_version: str = "reranker_v1"

    n_shards: int = 128
    final_k: int = 100
    metric_ks: tuple[int, ...] = (20, 50, 100, 200)
    optimize_metric: str = "ndcg@100"

    # Feature windows are strictly before the target window.
    feature_windows: tuple[int, ...] = (7, 14, 30, 60)
    feature_levels: tuple[str, ...] = ("user", "item", "category", "user_category", "user_item")
    candidate_scoped_feature_levels: tuple[str, ...] = ("user_item",)
    candidate_scoped_shard_batch_size: int = 8

    # Build-time fit dataset downsampling. Existing train_fit/val_fit can be reused; the memory-safe reader below
    # additionally samples rows before converting to pandas.
    always_keep_top_n_candidates: int = 50
    max_random_negatives_per_user: int = 120
    filter_fit_groups_without_positive: bool = True

    # Memory-safe model matrix controls.
    # Previous failure: 19.3M rows * 566 float32 features ~= 43.7GB just for one numeric block.
    model_max_numeric_features: int | None = 180
    model_keep_feature_windows: tuple[int, ...] = (7, 14, 30, 60)
    model_train_max_rows: int | None = 4_000_000
    model_val_max_rows: int | None = 1_200_000
    model_fit_hash_mod: int = 1_000_003
    model_force_all_positives: bool = True
    refit_production_on_train_val: bool = False  # False avoids a second large pandas concat; selection model is used for production.

    # Debug switches: set to an integer for a quick local run.
    sample_train_users: int | None = None
    sample_val_users: int | None = None
    sample_test_users: int | None = None

    # Model selection. CatBoost был хуже/дороже в текущих экспериментах, поэтому выключен по умолчанию.
    run_lightgbm: bool = True
    run_catboost: bool = False
    optuna_trials: int = 12
    catboost_optuna_trials: int = 4
    optuna_timeout_sec: int | None = None
    early_stopping_rounds: int = 80
    random_state: int = 42
    catboost_use_id_features: bool = False

    # Rebuild controls.
    rebuild_features: bool = False
    rebuild_datasets: bool = False
    rebuild_models: bool = False

    # Submission/test output controls.
    fill_missing_test_users_with_top_popular: bool = True
    fallback_pool_size: int = 5000

RCFG = FinalRankerConfig()

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
DATAMART_DIR = DATA_DIR / "datamart"
AGG_DIR = DATAMART_DIR / "agg"
CANDIDATES_DIR = DATAMART_DIR / "candidates"
RERANKED_ROOT = CANDIDATES_DIR / "reranked"
PROCESSED_ROOT = DATA_DIR / "processed" / "reranker"
FEATURES_DIR = DATAMART_DIR / "features" / RCFG.feature_store_version
MODELS_ROOT = PROJECT_ROOT / "models" / "reranker"
REPORTS_ROOT = PROJECT_ROOT / "artifacts" / "reports"

for path in [CANDIDATES_DIR, RERANKED_ROOT, PROCESSED_ROOT, FEATURES_DIR, MODELS_ROOT, REPORTS_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print(json.dumps({k: str(v) if isinstance(v, date) else v for k, v in asdict(RCFG).items()}, ensure_ascii=False, indent=2))


PROJECT_ROOT: c:\Users\denis\Documents\Repo\ozon
{
  "train_context_start": "2025-04-18",
  "train_context_end": "2025-06-03",
  "train_target_start": "2025-06-04",
  "train_target_end": "2025-06-17",
  "val_context_start": "2025-04-18",
  "val_context_end": "2025-06-17",
  "val_target_start": "2025-06-18",
  "val_target_end": "2025-07-01",
  "full_context_start": "2025-05-02",
  "full_context_end": "2025-07-01",
  "test_target_start": "2025-07-02",
  "test_target_end": "2025-07-15",
  "test_has_labels": true,
  "candidate_input_name": "merged_with_dl_all",
  "fallback_candidate_input_names": [
    "merged_with_bert4rec_small",
    "merged_with_sasrec_v3",
    "merged"
  ],
  "run_name": "final_reranker_lgbm_dl",
  "feature_store_version": "reranker_v1",
  "n_shards": 128,
  "final_k": 100,
  "metric_ks": [
    20,
    50,
    100,
    200
  ],
  "optimize_metric": "ndcg@100",
  "feature_windows": [
    7,
    14,
    30,
    60
  ],
  "feature_levels": [
    "user",
    "item",
    "c

In [3]:
EVENT_METRIC_COLS = (
    "cnt_events",
    "cnt_page_view",
    "cnt_view_description",
    "cnt_review_view",
    "cnt_to_cart",
    "cnt_favorite",
    "cnt_remove",
    "cnt_unfavorite",
)

ORDER_METRIC_COLS = (
    "cnt_orders",
    "cnt_delivered_orders",
    "cnt_canceled_orders",
    "cnt_proccesed_orders",
)

LEVEL_KEYS = {
    "user": ["user_id"],
    "item": ["item_id"],
    "user_item": ["user_id", "item_id"],
    "user_category": ["user_id", "catalogid"],
    "category": ["catalogid"],
}

_NUMERIC_DTYPES = {
    pl.Int8, pl.Int16, pl.Int32, pl.Int64,
    pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64,
    pl.Float32, pl.Float64,
    pl.Boolean,
}


def sanitize_name(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value)).strip("_") or "default"


def path_has_parquet(path: Path) -> bool:
    return path.exists() and any(path.rglob("*.parquet"))


def scan_parquet(path: Path | str, *, hive_partitioning: bool = True) -> pl.LazyFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.is_dir():
        if not path_has_parquet(path):
            raise FileNotFoundError(f"No parquet files found in {path}")
        return pl.scan_parquet(str(path / "**" / "*.parquet"), hive_partitioning=hive_partitioning)
    return pl.scan_parquet(str(path), hive_partitioning=hive_partitioning)


def collect_streaming(lf: pl.LazyFrame) -> pl.DataFrame:
    try:
        return lf.collect(engine="streaming")
    except TypeError:
        return lf.collect(streaming=True)


def lazy_columns(lf: pl.LazyFrame) -> list[str]:
    try:
        return list(lf.collect_schema().names())
    except Exception:
        return list(lf.schema.keys())


def ensure_numeric_cols(lf: pl.LazyFrame, cols: Iterable[str], dtype=pl.Float32) -> pl.LazyFrame:
    existing = set(lazy_columns(lf))
    missing_exprs = [pl.lit(0).cast(dtype).alias(c) for c in cols if c not in existing]
    return lf.with_columns(missing_exprs) if missing_exprs else lf


def clean_output_dir(path: Path) -> None:
    if path.exists():
        if path.is_dir():
            shutil.rmtree(path)
        else:
            path.unlink()
    path.mkdir(parents=True, exist_ok=True)


def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def events_path(level: str) -> Path:
    return AGG_DIR / "events" / level


def orders_path(level: str) -> Path:
    return AGG_DIR / "orders" / level


def _date_filter(lf: pl.LazyFrame, date_col: str, start: date, end: date) -> pl.LazyFrame:
    return (
        lf.with_columns(pl.col(date_col).cast(pl.Date, strict=False).alias(date_col))
        .filter(
            (pl.col(date_col) >= pl.lit(start).cast(pl.Date))
            & (pl.col(date_col) <= pl.lit(end).cast(pl.Date))
        )
    )


def read_events_lf(level: str, start: date, end: date) -> pl.LazyFrame:
    lf = scan_parquet(events_path(level), hive_partitioning=True)
    return _date_filter(lf, "event_date", start, end)


def read_orders_lf(level: str, start: date, end: date) -> pl.LazyFrame:
    lf = scan_parquet(orders_path(level), hive_partitioning=True)
    return _date_filter(lf, "created_date", start, end)


def candidate_files(path: Path) -> list[Path]:
    if not path.exists():
        return []
    return sorted(path.rglob("*.parquet"))


def scan_candidate_shard_or_none(candidates_dir: Path, shard_id: int) -> pl.LazyFrame | None:
    shard_dir = candidates_dir / f"user_shard={shard_id}"
    if not path_has_parquet(shard_dir):
        return None
    return scan_parquet(shard_dir, hive_partitioning=True)


def write_single_shard(df: pl.DataFrame, out_dir: Path, shard_id: int, filename: str = "part.parquet") -> int:
    shard_dir = out_dir / f"user_shard={int(shard_id)}"
    ensure_dir(shard_dir)
    path = shard_dir / filename
    df.write_parquet(path, compression="zstd")
    return int(df.height)


def rel(path: Path | str) -> str:
    path = Path(path)
    try:
        return str(path.resolve().relative_to(PROJECT_ROOT.resolve()))
    except Exception:
        return str(path)


def candidate_set_key(split_name: str) -> str:
    # Candidate-scoped features depend on candidate pool. Keep them separated from old base-reranker caches.
    return f"{sanitize_name(ACTIVE_CANDIDATE_INPUT_NAME)}_{sanitize_name(split_name)}"

In [4]:
def resolve_candidate_dir(split: str) -> tuple[Path, str]:
    tried: list[Path] = []
    for name in (RCFG.candidate_input_name, *RCFG.fallback_candidate_input_names):
        path = CANDIDATES_DIR / name / split
        tried.append(path)
        if path_has_parquet(path):
            return path, name
    tried_msg = "\n".join(f"- {rel(p)}" for p in tried)
    raise FileNotFoundError(
        f"Не найден candidate pool для split={split}. Проверенные пути:\n{tried_msg}\n"
        "Для финального ранкера сначала запусти DL_Merge_Experiments_4sources.ipynb "
        "с materialized_output_name='merged_with_dl_all'."
    )


train_candidates_dir, train_candidate_input_name = resolve_candidate_dir("ranker_train")
val_candidates_dir, val_candidate_input_name = resolve_candidate_dir("val")
test_candidates_dir, test_candidate_input_name = resolve_candidate_dir("test")

candidate_names = {train_candidate_input_name, val_candidate_input_name, test_candidate_input_name}
if len(candidate_names) != 1:
    raise RuntimeError(f"Разные candidate inputs по split'ам: {candidate_names}")

ACTIVE_CANDIDATE_INPUT_NAME = train_candidate_input_name
RUN_NAME = f"{sanitize_name(RCFG.run_name)}__{sanitize_name(ACTIVE_CANDIDATE_INPUT_NAME)}"
PROCESSED_DIR = PROCESSED_ROOT / RUN_NAME
DATASET_DIR = PROCESSED_DIR / "dataset"
MODELS_DIR = MODELS_ROOT / RUN_NAME
RERANKED_DIR = RERANKED_ROOT / RUN_NAME
REPORTS_DIR = REPORTS_ROOT / RUN_NAME

for path in [PROCESSED_DIR, DATASET_DIR, MODELS_DIR, RERANKED_DIR, REPORTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("ACTIVE_CANDIDATE_INPUT_NAME:", ACTIVE_CANDIDATE_INPUT_NAME)
print("train_candidates_dir:", rel(train_candidates_dir))
print("val_candidates_dir:  ", rel(val_candidates_dir))
print("test_candidates_dir: ", rel(test_candidates_dir))
print("RUN_NAME:", RUN_NAME)
print("DATASET_DIR:", rel(DATASET_DIR))
print("MODELS_DIR:", rel(MODELS_DIR))
print("REPORTS_DIR:", rel(REPORTS_DIR))

if ACTIVE_CANDIDATE_INPUT_NAME != RCFG.candidate_input_name:
    print(
        f"WARNING: основной финальный pool {RCFG.candidate_input_name!r} не найден; "
        f"используется fallback {ACTIVE_CANDIDATE_INPUT_NAME!r}."
    )

ACTIVE_CANDIDATE_INPUT_NAME: merged_with_dl_all
train_candidates_dir: data\datamart\candidates\merged_with_dl_all\ranker_train
val_candidates_dir:   data\datamart\candidates\merged_with_dl_all\val
test_candidates_dir:  data\datamart\candidates\merged_with_dl_all\test
RUN_NAME: final_reranker_lgbm_dl__merged_with_dl_all
DATASET_DIR: data\processed\reranker\final_reranker_lgbm_dl__merged_with_dl_all\dataset
MODELS_DIR: models\reranker\final_reranker_lgbm_dl__merged_with_dl_all
REPORTS_DIR: artifacts\reports\final_reranker_lgbm_dl__merged_with_dl_all


In [5]:
def build_ground_truth(start: date, end: date) -> dict[int, list[int]]:
    gt_df = collect_streaming(
        read_orders_lf("user_item", start, end)
        .select(
            pl.col("user_id").cast(pl.Int64),
            pl.col("item_id").cast(pl.Int64),
        )
        .drop_nulls()
        .unique()
        .group_by("user_id")
        .agg(pl.col("item_id").sort().alias("gt_items"))
        .sort("user_id")
    )
    return {int(u): [int(x) for x in items] for u, items in zip(gt_df["user_id"], gt_df["gt_items"])}


def sample_user_ids(user_ids: list[int], n: int | None, seed: int = RCFG.random_state) -> list[int]:
    user_ids = sorted(int(x) for x in user_ids)
    if n is None or n >= len(user_ids):
        return user_ids
    rng = np.random.default_rng(seed)
    sampled = rng.choice(np.asarray(user_ids, dtype=np.int64), size=n, replace=False)
    return sorted(int(x) for x in sampled)


def find_test_users_path() -> Path | None:
    candidates = [
        DATA_DIR / "ml_ozon_recsys_test.snappy.parquet",
        DATA_DIR / "ml_ozon_recsys_test.parquet",
        DATA_DIR / "test.parquet",
        RAW_DIR / "ml_ozon_recsys_test.snappy.parquet",
        RAW_DIR / "ml_ozon_recsys_test.parquet",
        RAW_DIR / "ml_ozon_recsys_test_final_apparel_orders_data",
        RAW_DIR / "ml_ozon_recsys_test_final_apparel_orders_data.parquet",
    ]
    for path in candidates:
        if path.exists():
            return path

    for root in [DATA_DIR, RAW_DIR]:
        if root.exists():
            matched = sorted(
                p for p in root.iterdir()
                if "test" in p.name.lower() and p.suffix.lower() in {".parquet", ".pq"}
            )
            if matched:
                return matched[0]
    return None


def load_test_users() -> list[int]:
    test_path = find_test_users_path()
    if test_path is not None:
        print("test users path:", test_path)
        df = collect_streaming(
            scan_parquet(test_path, hive_partitioning=True)
            .select(pl.col("user_id").cast(pl.Int64))
            .drop_nulls()
            .unique()
            .sort("user_id")
        )
        return [int(x) for x in df["user_id"].to_numpy()]

    print("Test users file not found. Fallback: users from datamart orders in test target window.")
    return sorted(build_ground_truth(RCFG.test_target_start, RCFG.test_target_end).keys())


train_gt = build_ground_truth(RCFG.train_target_start, RCFG.train_target_end)
val_gt = build_ground_truth(RCFG.val_target_start, RCFG.val_target_end)
test_gt = build_ground_truth(RCFG.test_target_start, RCFG.test_target_end) if RCFG.test_has_labels else {}

test_users_all = load_test_users()

train_users = sample_user_ids(sorted(train_gt.keys()), RCFG.sample_train_users)
val_users = sample_user_ids(sorted(val_gt.keys()), RCFG.sample_val_users)
test_users = sample_user_ids(test_users_all, RCFG.sample_test_users)

# Для финальных test-метрик оцениваем только тех test users, у кого есть ground truth в test target window
# и кто попал в текущий test dataset/user sample. Остальные users все равно попадут в recommendations/fallback.
test_user_set = set(map(int, test_users))
test_eval_gt = {int(u): items for u, items in test_gt.items() if int(u) in test_user_set}

print(f"train users with GT: {len(train_users):,}")
print(f"val users with GT:   {len(val_users):,}")
print(f"test users total:    {len(test_users):,}")
print(f"test users with GT:  {len(test_eval_gt):,}")
print(f"avg train GT size:   {np.mean([len(train_gt[u]) for u in train_users]):.3f}")
print(f"avg val GT size:     {np.mean([len(val_gt[u]) for u in val_users]):.3f}")
if test_eval_gt:
    print(f"avg test GT size:    {np.mean([len(v) for v in test_eval_gt.values()]):.3f}")
elif RCFG.test_has_labels:
    warnings.warn("RCFG.test_has_labels=True, но test_eval_gt пустой. Проверь test users и окно test target.")


Test users file not found. Fallback: users from datamart orders in test target window.
train users with GT: 376,016
val users with GT:   359,928
test users total:    346,759
test users with GT:  346,759
avg train GT size:   4.626
avg val GT size:     4.381
avg test GT size:    4.119


## Метрики

Метрики считаются по пользователям с ground truth в target-окне. `val_eval` используется для выбора модели и гиперпараметров. `test` в этом проекте — labeled holdout: на нем считаются финальные метрики один раз, без подбора модели по test.

In [6]:
def _unique_int_list(values) -> list[int]:
    if values is None:
        return []
    return list(dict.fromkeys(int(x) for x in values))


def user_metrics_at_k(gt_items: Sequence[int], pred_items: Sequence[int], k: int) -> dict[str, float]:
    gt = set(_unique_int_list(gt_items))
    if not gt:
        return {
            "hit_rate": np.nan,
            "recall": np.nan,
            "precision": np.nan,
            "ndcg": np.nan,
            "map": np.nan,
            "mrr": np.nan,
        }

    pred = _unique_int_list(pred_items)[:k]
    hits = [1 if item in gt else 0 for item in pred]

    hit_rate = float(any(hits))
    recall = float(sum(hits) / len(gt))
    precision = float(sum(hits) / k) if k > 0 else 0.0

    dcg = sum(h / np.log2(rank + 2.0) for rank, h in enumerate(hits))
    idcg = sum(1.0 / np.log2(rank + 2.0) for rank in range(min(len(gt), k)))
    ndcg = float(dcg / idcg) if idcg > 0 else 0.0

    hit_count = 0
    ap_sum = 0.0
    rr = 0.0
    for rank, h in enumerate(hits, start=1):
        if h:
            hit_count += 1
            ap_sum += hit_count / rank
            if rr == 0.0:
                rr = 1.0 / rank
    ap = float(ap_sum / len(gt))

    return {
        "hit_rate": hit_rate,
        "recall": recall,
        "precision": precision,
        "ndcg": ndcg,
        "map": ap,
        "mrr": float(rr),
    }


def evaluate_predictions_by_user(
    preds_by_user: dict[int, list[int]],
    ground_truth: dict[int, list[int]],
    *,
    ks: Iterable[int] = RCFG.metric_ks,
    label: str = "model",
) -> pd.DataFrame:
    rows = []
    gt_users = set(int(u) for u in ground_truth)

    for k in ks:
        sums = defaultdict(float)
        n_eval = 0

        for user_id, gt_items in ground_truth.items():
            user_id = int(user_id)
            pred_items = preds_by_user.get(user_id, [])
            m = user_metrics_at_k(gt_items, pred_items, int(k))
            if np.isnan(m["ndcg"]):
                continue
            n_eval += 1
            for metric, value in m.items():
                sums[metric] += float(value)

        row = {
            "model": label,
            "k": int(k),
            "users_total": len(gt_users),
            "users_with_predictions": len(set(preds_by_user) & gt_users),
            "user_coverage": len(set(preds_by_user) & gt_users) / len(gt_users) if gt_users else np.nan,
        }
        for metric in ["hit_rate", "recall", "precision", "ndcg", "map", "mrr"]:
            row[metric] = sums[metric] / n_eval if n_eval else np.nan
        rows.append(row)

    return pd.DataFrame(rows)


def evaluate_scored_df(
    df: pl.DataFrame | pd.DataFrame,
    ground_truth: dict[int, list[int]],
    *,
    score_col: str = "score",
    label: str = "model",
    ks: Iterable[int] = RCFG.metric_ks,
    higher_is_better: bool = True,
) -> pd.DataFrame:
    if isinstance(df, pl.DataFrame):
        pdf = df.select(["user_id", "item_id", score_col]).to_pandas()
    else:
        pdf = df[["user_id", "item_id", score_col]].copy()

    pdf = pdf.sort_values(["user_id", score_col, "item_id"], ascending=[True, not higher_is_better, True])
    preds_by_user = {
        int(user_id): group["item_id"].astype(int).tolist()
        for user_id, group in pdf.groupby("user_id", sort=False)
    }
    return evaluate_predictions_by_user(preds_by_user, ground_truth, ks=ks, label=label)


def evaluate_dataset_score_col(
    dataset_dir: Path,
    ground_truth: dict[int, list[int]],
    *,
    score_col: str,
    label: str,
    higher_is_better: bool = True,
    ks: Iterable[int] = RCFG.metric_ks,
) -> pd.DataFrame:
    preds_by_user: dict[int, list[int]] = {}
    files = candidate_files(dataset_dir)
    for file in tqdm(files, desc=f"Evaluate {label}"):
        part = pl.read_parquet(file, columns=["user_id", "item_id", score_col])
        part = part.sort(["user_id", score_col, "item_id"], descending=[False, higher_is_better, False])
        for row in part.group_by("user_id", maintain_order=True).agg(pl.col("item_id").alias("items")).iter_rows(named=True):
            preds_by_user[int(row["user_id"])] = [int(x) for x in row["items"]]
    return evaluate_predictions_by_user(preds_by_user, ground_truth, ks=ks, label=label)


def evaluate_scored_dir(
    scored_dir: Path,
    ground_truth: dict[int, list[int]],
    *,
    rank_col: str = "rank",
    label: str = "model",
    ks: Iterable[int] = RCFG.metric_ks,
) -> pd.DataFrame:
    preds_by_user: dict[int, list[int]] = {}
    files = candidate_files(scored_dir)
    for file in tqdm(files, desc=f"Evaluate {label}"):
        part = pl.read_parquet(file, columns=["user_id", "item_id", rank_col])
        part = part.sort(["user_id", rank_col, "item_id"], descending=[False, False, False])
        for row in part.group_by("user_id", maintain_order=True).agg(pl.col("item_id").alias("items")).iter_rows(named=True):
            preds_by_user[int(row["user_id"])] = [int(x) for x in row["items"]]
    return evaluate_predictions_by_user(preds_by_user, ground_truth, ks=ks, label=label)

## Point-in-time feature store

Common уровни `user/item/category/user_category` переиспользуются по `as_of_date`. Высококардинальный `user_item` строится только для candidate pairs текущего candidate pool, чтобы не материализовать полную матрицу user-item.

In [7]:
def safe_ratio_expr(num_col: str, den_col: str, out_col: str) -> pl.Expr:
    num = pl.col(num_col).cast(pl.Float32).fill_null(0.0)
    den = pl.col(den_col).cast(pl.Float32).fill_null(0.0)
    return (
        pl.when(den > 0.0)
        .then(num / den)
        .otherwise(pl.lit(None))
        .cast(pl.Float32)
        .alias(out_col)
    )


def item_catalog_lf() -> pl.LazyFrame:
    """
    item_id -> catalogid mapping. It is needed both for global category features and for
    candidate-scoped user_category features.
    """
    path = DATAMART_DIR / "mappings" / "item_catalog.parquet"
    if path.exists():
        return scan_parquet(path, hive_partitioning=False).select(
            pl.col("item_id").cast(pl.Int64),
            pl.col("catalogid").cast(pl.Int64),
        ).unique()

    # Fallback: infer mapping from available user_item aggregates.
    for candidate in [events_path("user_item"), orders_path("user_item")]:
        if path_has_parquet(candidate):
            lf = scan_parquet(candidate, hive_partitioning=True)
            cols = lazy_columns(lf)
            if "catalogid" in cols:
                return lf.select(
                    pl.col("item_id").cast(pl.Int64),
                    pl.col("catalogid").cast(pl.Int64),
                ).drop_nulls().unique()

    raise FileNotFoundError(
        "Не найден item->catalogid mapping. Ожидался data/datamart/mappings/item_catalog.parquet "
        "или catalogid в agg/events/user_item | agg/orders/user_item."
    )


def source_lf_for_level(domain: str, level: str, start: date, end: date) -> pl.LazyFrame | None:
    if domain == "events":
        path_fn = events_path
        read_fn = read_events_lf
    elif domain == "orders":
        path_fn = orders_path
        read_fn = read_orders_lf
    else:
        raise ValueError(domain)

    if path_has_parquet(path_fn(level)):
        return read_fn(level, start, end)

    # Fallback from user_item if derived aggs were not built.
    if level != "user_item" and path_has_parquet(path_fn("user_item")):
        return read_fn("user_item", start, end)

    return None


def prepare_source_keys_for_level(lf: pl.LazyFrame, level: str) -> pl.LazyFrame:
    """Ensure keys required by `level` exist and have stable integer dtypes."""
    cols = set(lazy_columns(lf))

    if "catalogid" in LEVEL_KEYS[level] and "catalogid" not in cols:
        if "item_id" not in cols:
            raise ValueError(f"Cannot build {level}: source has neither catalogid nor item_id")
        lf = lf.join(item_catalog_lf(), on="item_id", how="left")
        cols.add("catalogid")

    casts = []
    for key in LEVEL_KEYS[level]:
        if key not in cols and key not in lazy_columns(lf):
            raise ValueError(f"Cannot build {level}: missing key column {key}")
        casts.append(pl.col(key).cast(pl.Int64, strict=False).alias(key))
    return lf.with_columns(casts)


def aggregate_events_level(level: str, start: date, end: date, window: int) -> pl.LazyFrame | None:
    keys = LEVEL_KEYS[level]
    lf = source_lf_for_level("events", level, start, end)
    if lf is None:
        return None

    lf = prepare_source_keys_for_level(lf, level)
    lf = ensure_numeric_cols(lf, EVENT_METRIC_COLS)

    prefix = f"f_num__{level}__"
    agg_exprs = [
        pl.sum(c).cast(pl.Float32).alias(f"{prefix}{c}_{window}d")
        for c in EVENT_METRIC_COLS
    ]
    agg_exprs.append(pl.max("event_date").alias("__last_event_date"))

    out = lf.group_by(keys).agg(agg_exprs)

    cnt_events = f"{prefix}cnt_events_{window}d"
    cnt_page = f"{prefix}cnt_page_view_{window}d"
    cnt_desc = f"{prefix}cnt_view_description_{window}d"
    cnt_review = f"{prefix}cnt_review_view_{window}d"
    cnt_cart = f"{prefix}cnt_to_cart_{window}d"
    cnt_fav = f"{prefix}cnt_favorite_{window}d"

    out = out.with_columns(
        (pl.lit(end).cast(pl.Date) - pl.col("__last_event_date").cast(pl.Date))
        .dt.total_days()
        .cast(pl.Float32)
        .alias(f"{prefix}days_since_last_event_{window}d"),
        (pl.col(cnt_events).fill_null(0.0) > 0).cast(pl.Int8).alias(f"{prefix}has_events_{window}d"),
        safe_ratio_expr(cnt_cart, cnt_page, f"{prefix}to_cart_per_page_view_{window}d"),
        safe_ratio_expr(cnt_fav, cnt_page, f"{prefix}favorite_per_page_view_{window}d"),
        safe_ratio_expr(cnt_desc, cnt_page, f"{prefix}description_per_page_view_{window}d"),
        safe_ratio_expr(cnt_review, cnt_page, f"{prefix}review_per_page_view_{window}d"),
    ).drop("__last_event_date")

    return out


def aggregate_orders_level(level: str, start: date, end: date, window: int) -> pl.LazyFrame | None:
    keys = LEVEL_KEYS[level]
    lf = source_lf_for_level("orders", level, start, end)
    if lf is None:
        return None

    lf = prepare_source_keys_for_level(lf, level)
    lf = ensure_numeric_cols(lf, ORDER_METRIC_COLS)

    prefix = f"f_num__{level}__"
    agg_exprs = [
        pl.sum(c).cast(pl.Float32).alias(f"{prefix}{c}_{window}d")
        for c in ORDER_METRIC_COLS
    ]
    agg_exprs.append(pl.max("created_date").alias("__last_order_date"))

    out = lf.group_by(keys).agg(agg_exprs)

    cnt_orders = f"{prefix}cnt_orders_{window}d"
    cnt_delivered = f"{prefix}cnt_delivered_orders_{window}d"
    cnt_cancel = f"{prefix}cnt_canceled_orders_{window}d"
    cnt_processed = f"{prefix}cnt_proccesed_orders_{window}d"

    out = out.with_columns(
        (pl.lit(end).cast(pl.Date) - pl.col("__last_order_date").cast(pl.Date))
        .dt.total_days()
        .cast(pl.Float32)
        .alias(f"{prefix}days_since_last_order_{window}d"),
        (pl.col(cnt_orders).fill_null(0.0) > 0).cast(pl.Int8).alias(f"{prefix}has_orders_{window}d"),
        safe_ratio_expr(cnt_delivered, cnt_orders, f"{prefix}delivered_share_{window}d"),
        safe_ratio_expr(cnt_cancel, cnt_orders, f"{prefix}cancel_share_{window}d"),
        safe_ratio_expr(cnt_processed, cnt_orders, f"{prefix}processed_share_{window}d"),
    ).drop("__last_order_date")

    return out


def feature_date_dir(as_of_date: date) -> Path:
    return FEATURES_DIR / f"as_of_date={as_of_date.isoformat()}"


def build_level_features_for_split(split_name: str, context_start: date, context_end: date, level: str) -> Path | None:
    # split_name is used only for logs. The physical feature partition is date-based.
    out_dir = feature_date_dir(context_end)
    ensure_dir(out_dir)
    out_path = out_dir / f"{level}.parquet"

    if out_path.exists() and not RCFG.rebuild_features:
        return out_path

    parts: list[pl.LazyFrame] = []

    for window in RCFG.feature_windows:
        win_start = max(context_start, context_end - timedelta(days=int(window) - 1))

        ev = aggregate_events_level(level, win_start, context_end, int(window))
        if ev is not None:
            parts.append(ev)

        orders = aggregate_orders_level(level, win_start, context_end, int(window))
        if orders is not None:
            parts.append(orders)

    if not parts:
        print(f"{split_name}/{level}: no feature parts")
        return None

    keys = LEVEL_KEYS[level]
    base_keys = pl.concat([p.select(keys) for p in parts], how="vertical_relaxed").unique()

    features_lf = base_keys
    for part in parts:
        features_lf = features_lf.join(part, on=keys, how="left")

    df = collect_streaming(features_lf)
    if df.is_empty():
        print(f"{split_name}/{level}: empty")
        return None

    df.write_parquet(out_path, compression="zstd")
    print(f"{split_name}/{level}: rows={df.height:,}, cols={df.width:,} -> {rel(out_path)}")
    del df
    gc.collect()
    return out_path


def scan_candidate_keys_for_shards(candidates_dir: Path, level: str, shard_ids: Sequence[int]) -> pl.LazyFrame | None:
    """
    Read only keys that may be joined to candidates. This is the core optimization:
    high-cardinality features are computed for candidate pairs/categories, not for all historical pairs.
    """
    keys = LEVEL_KEYS[level]
    parts: list[pl.LazyFrame] = []

    for shard_id in shard_ids:
        cand_lf = scan_candidate_shard_or_none(candidates_dir, int(shard_id))
        if cand_lf is None:
            continue

        cand_lf = cand_lf.select(
            pl.col("user_id").cast(pl.Int64),
            pl.col("item_id").cast(pl.Int64),
        )

        if level == "user_item":
            key_lf = cand_lf.select(keys)
        elif level == "user_category":
            key_lf = (
                cand_lf.join(item_catalog_lf(), on="item_id", how="left")
                .with_columns(pl.col("catalogid").fill_null(-1).cast(pl.Int64).alias("catalogid"))
                .select(keys)
            )
        else:
            raise ValueError(f"Candidate-scoped build is intended only for high-cardinality levels, got {level}")

        parts.append(key_lf)

    if not parts:
        return None

    return pl.concat(parts, how="vertical_relaxed").unique()


def restrict_to_candidate_keys(lf: pl.LazyFrame, level: str, candidate_keys_lf: pl.LazyFrame) -> pl.LazyFrame:
    keys = LEVEL_KEYS[level]
    lf = prepare_source_keys_for_level(lf, level)
    candidate_keys_lf = candidate_keys_lf.select(keys).unique()
    return lf.join(candidate_keys_lf, on=keys, how="inner")


def aggregate_events_level_candidate_scoped(
    level: str,
    candidate_keys_lf: pl.LazyFrame,
    start: date,
    end: date,
    window: int,
) -> pl.LazyFrame | None:
    lf = source_lf_for_level("events", level, start, end)
    if lf is None:
        return None
    lf = restrict_to_candidate_keys(lf, level, candidate_keys_lf)
    lf = ensure_numeric_cols(lf, EVENT_METRIC_COLS)

    keys = LEVEL_KEYS[level]
    prefix = f"f_num__{level}__"
    agg_exprs = [pl.sum(c).cast(pl.Float32).alias(f"{prefix}{c}_{window}d") for c in EVENT_METRIC_COLS]
    agg_exprs.append(pl.max("event_date").alias("__last_event_date"))

    out = lf.group_by(keys).agg(agg_exprs)

    cnt_events = f"{prefix}cnt_events_{window}d"
    cnt_page = f"{prefix}cnt_page_view_{window}d"
    cnt_desc = f"{prefix}cnt_view_description_{window}d"
    cnt_review = f"{prefix}cnt_review_view_{window}d"
    cnt_cart = f"{prefix}cnt_to_cart_{window}d"
    cnt_fav = f"{prefix}cnt_favorite_{window}d"

    return out.with_columns(
        (pl.lit(end).cast(pl.Date) - pl.col("__last_event_date").cast(pl.Date))
        .dt.total_days()
        .cast(pl.Float32)
        .alias(f"{prefix}days_since_last_event_{window}d"),
        (pl.col(cnt_events).fill_null(0.0) > 0).cast(pl.Int8).alias(f"{prefix}has_events_{window}d"),
        safe_ratio_expr(cnt_cart, cnt_page, f"{prefix}to_cart_per_page_view_{window}d"),
        safe_ratio_expr(cnt_fav, cnt_page, f"{prefix}favorite_per_page_view_{window}d"),
        safe_ratio_expr(cnt_desc, cnt_page, f"{prefix}description_per_page_view_{window}d"),
        safe_ratio_expr(cnt_review, cnt_page, f"{prefix}review_per_page_view_{window}d"),
    ).drop("__last_event_date")


def aggregate_orders_level_candidate_scoped(
    level: str,
    candidate_keys_lf: pl.LazyFrame,
    start: date,
    end: date,
    window: int,
) -> pl.LazyFrame | None:
    lf = source_lf_for_level("orders", level, start, end)
    if lf is None:
        return None
    lf = restrict_to_candidate_keys(lf, level, candidate_keys_lf)
    lf = ensure_numeric_cols(lf, ORDER_METRIC_COLS)

    keys = LEVEL_KEYS[level]
    prefix = f"f_num__{level}__"
    agg_exprs = [pl.sum(c).cast(pl.Float32).alias(f"{prefix}{c}_{window}d") for c in ORDER_METRIC_COLS]
    agg_exprs.append(pl.max("created_date").alias("__last_order_date"))

    out = lf.group_by(keys).agg(agg_exprs)

    cnt_orders = f"{prefix}cnt_orders_{window}d"
    cnt_delivered = f"{prefix}cnt_delivered_orders_{window}d"
    cnt_cancel = f"{prefix}cnt_canceled_orders_{window}d"
    cnt_processed = f"{prefix}cnt_proccesed_orders_{window}d"

    return out.with_columns(
        (pl.lit(end).cast(pl.Date) - pl.col("__last_order_date").cast(pl.Date))
        .dt.total_days()
        .cast(pl.Float32)
        .alias(f"{prefix}days_since_last_order_{window}d"),
        (pl.col(cnt_orders).fill_null(0.0) > 0).cast(pl.Int8).alias(f"{prefix}has_orders_{window}d"),
        safe_ratio_expr(cnt_delivered, cnt_orders, f"{prefix}delivered_share_{window}d"),
        safe_ratio_expr(cnt_cancel, cnt_orders, f"{prefix}cancel_share_{window}d"),
        safe_ratio_expr(cnt_processed, cnt_orders, f"{prefix}processed_share_{window}d"),
    ).drop("__last_order_date")


def write_feature_batch_by_user_shard(df: pl.DataFrame, out_dir: Path, shard_ids: Sequence[int]) -> int:
    if df.is_empty():
        return 0

    if "user_id" not in df.columns:
        raise ValueError("Candidate-scoped features must contain user_id for sharded write")

    df = df.with_columns((pl.col("user_id").cast(pl.Int64) % int(RCFG.n_shards)).cast(pl.Int32).alias("user_shard"))
    rows_written = 0
    for shard_id in shard_ids:
        part = df.filter(pl.col("user_shard") == int(shard_id)).drop("user_shard")
        if part.is_empty():
            continue
        rows_written += write_single_shard(part, out_dir, int(shard_id), "features.parquet")
    return rows_written


def build_candidate_scoped_level_features_for_split(
    split_name: str,
    candidates_dir: Path,
    context_start: date,
    context_end: date,
    level: str,
) -> Path | None:
    """
    Optimized high-cardinality feature builder.

    Instead of materializing all historical user_item/user_category combinations, it:
    1. reads candidate keys for a small batch of user shards;
    2. filters historical events/orders to these keys via inner join;
    3. aggregates only the surviving rows;
    4. writes features back partitioned by user_shard.
    """
    # Candidate-scoped interaction features are date-based, but also depend on the candidate set.
    # candidate_set is kept in the path because the keys are limited to this particular pool.
    out_dir = FEATURES_DIR / "candidate_scoped" / f"as_of_date={context_end.isoformat()}" / f"candidate_set={candidate_set_key(split_name)}" / level

    if path_has_parquet(out_dir) and not RCFG.rebuild_features:
        return out_dir

    clean_output_dir(out_dir)
    total_rows = 0
    batch_size = max(1, int(RCFG.candidate_scoped_shard_batch_size))
    all_shards = list(range(int(RCFG.n_shards)))

    for pos in trange(0, len(all_shards), batch_size, desc=f"{split_name}/{level} candidate-scoped"):
        shard_ids = all_shards[pos:pos + batch_size]
        candidate_keys_lf = scan_candidate_keys_for_shards(candidates_dir, level, shard_ids)
        if candidate_keys_lf is None:
            continue

        parts: list[pl.LazyFrame] = []
        for window in RCFG.feature_windows:
            win_start = max(context_start, context_end - timedelta(days=int(window) - 1))

            ev = aggregate_events_level_candidate_scoped(level, candidate_keys_lf, win_start, context_end, int(window))
            if ev is not None:
                parts.append(ev)

            orders = aggregate_orders_level_candidate_scoped(level, candidate_keys_lf, win_start, context_end, int(window))
            if orders is not None:
                parts.append(orders)

        if not parts:
            continue

        keys = LEVEL_KEYS[level]
        base_keys = pl.concat([p.select(keys) for p in parts], how="vertical_relaxed").unique()
        features_lf = base_keys
        for part in parts:
            features_lf = features_lf.join(part, on=keys, how="left")

        df = collect_streaming(features_lf)
        if df.is_empty():
            del df
            gc.collect()
            continue

        rows_written = write_feature_batch_by_user_shard(df, out_dir, shard_ids)
        total_rows += rows_written
        print(f"{split_name}/{level}: shards={shard_ids[0]}..{shard_ids[-1]}, rows={rows_written:,}, cols={df.width:,}")
        del df
        gc.collect()

    if total_rows == 0:
        print(f"{split_name}/{level}: no candidate-scoped features")
        return None

    print(f"{split_name}/{level}: total_rows={total_rows:,} -> {rel(out_dir)}")
    return out_dir


def build_feature_store_for_split(
    split_name: str,
    context_start: date,
    context_end: date,
    candidates_dir: Path,
) -> dict[str, Path]:
    paths: dict[str, Path] = {}
    candidate_scoped_levels = set(RCFG.candidate_scoped_feature_levels)

    for level in RCFG.feature_levels:
        if level not in LEVEL_KEYS:
            raise ValueError(f"Unknown feature level: {level}. Available: {sorted(LEVEL_KEYS)}")

        if level in candidate_scoped_levels:
            path = build_candidate_scoped_level_features_for_split(
                split_name,
                candidates_dir,
                context_start,
                context_end,
                level,
            )
        else:
            path = build_level_features_for_split(split_name, context_start, context_end, level)

        if path is not None:
            paths[level] = path
    return paths

## Labels, candidate schema и ranker dataset

Этот блок делает ранкер устойчивым к разным candidate pool'ам: базовый `merged`, `merged_with_bert4rec_small`, `merged_with_dl_all` и т.д. Все source-specific `*_rank`, `*_score`, `in_*` автоматически превращаются в числовые признаки `f_num__candidate__...`.

In [8]:
def label_file_stats(path: Path) -> dict[str, int]:
    if not path.exists():
        return {"rows": 0, "positives": 0}
    df = collect_streaming(
        scan_parquet(path, hive_partitioning=False)
        .select(
            pl.len().alias("rows"),
            pl.col("label").fill_null(0).sum().alias("positives"),
        )
    )
    row = df.row(0, named=True)
    return {"rows": int(row["rows"] or 0), "positives": int(row["positives"] or 0)}


def build_labels_for_target(split_name: str, start: date, end: date, *, has_labels: bool) -> Path:
    out_dir = PROCESSED_DIR / "labels"
    ensure_dir(out_dir)
    out_path = out_dir / f"labels_{split_name}_{start.isoformat()}_{end.isoformat()}.parquet"

    schema = {
        "user_id": pl.Int64,
        "item_id": pl.Int64,
        "label": pl.UInt8,
        "target_order_cnt": pl.UInt16,
        "target_delivered_order_cnt": pl.UInt16,
        "target_canceled_order_cnt": pl.UInt16,
    }

    if out_path.exists() and not RCFG.rebuild_datasets:
        if not has_labels:
            return out_path

        stats = label_file_stats(out_path)
        if stats["rows"] > 0 and stats["positives"] > 0:
            return out_path

        print(
            f"{split_name}: existing labels are empty/unlabeled "
            f"(rows={stats['rows']:,}, positives={stats['positives']:,}) -> rebuild labels"
        )
        out_path.unlink(missing_ok=True)

    if not has_labels:
        pl.DataFrame(schema=schema).write_parquet(out_path, compression="zstd")
        return out_path

    lf = read_orders_lf("user_item", start, end)
    lf = ensure_numeric_cols(lf, ORDER_METRIC_COLS, dtype=pl.Float32)
    labels = collect_streaming(
        lf.select(
            pl.col("user_id").cast(pl.Int64),
            pl.col("item_id").cast(pl.Int64),
            pl.col("cnt_orders").fill_null(0).cast(pl.Float32),
            pl.col("cnt_delivered_orders").fill_null(0).cast(pl.Float32),
            pl.col("cnt_canceled_orders").fill_null(0).cast(pl.Float32),
        )
        .drop_nulls(["user_id", "item_id"])
        .group_by(["user_id", "item_id"])
        .agg(
            pl.sum("cnt_orders").alias("target_order_cnt_f"),
            pl.sum("cnt_delivered_orders").alias("target_delivered_order_cnt_f"),
            pl.sum("cnt_canceled_orders").alias("target_canceled_order_cnt_f"),
        )
        .with_columns(
            (pl.col("target_order_cnt_f") > 0).cast(pl.UInt8).alias("label"),
            pl.col("target_order_cnt_f").clip(0, 65535).cast(pl.UInt16).alias("target_order_cnt"),
            pl.col("target_delivered_order_cnt_f").clip(0, 65535).cast(pl.UInt16).alias("target_delivered_order_cnt"),
            pl.col("target_canceled_order_cnt_f").clip(0, 65535).cast(pl.UInt16).alias("target_canceled_order_cnt"),
        )
        .select(list(schema))
    )
    labels.write_parquet(out_path, compression="zstd")
    print(f"{split_name}: labels rows={labels.height:,}, positives={int(labels['label'].sum()):,} -> {rel(out_path)}")
    return out_path


def reciprocal_rank_expr(col: str, out_col: str) -> pl.Expr:
    x = pl.col(col).cast(pl.Float32, strict=False)
    return (
        pl.when(x.is_not_null() & (x > 0.0))
        .then(1.0 / x)
        .otherwise(0.0)
        .cast(pl.Float32)
        .alias(out_col)
    )


def log1p_nonnegative_expr(col: str, out_col: str) -> pl.Expr:
    x = pl.col(col).cast(pl.Float32, strict=False).fill_null(0.0)
    x = pl.when(x > 0.0).then(x).otherwise(0.0)
    return (x + 1.0).log().cast(pl.Float32).alias(out_col)


def ensure_candidate_schema(lf: pl.LazyFrame) -> pl.LazyFrame:
    cols = set(lazy_columns(lf))
    exprs: list[pl.Expr] = []

    if "candidate_rank" not in cols:
        if "rank" in cols:
            exprs.append(pl.col("rank").cast(pl.Int32, strict=False).alias("candidate_rank"))
        else:
            raise ValueError("Candidate shard must contain candidate_rank or rank")

    # Base non-DL source columns.
    defaults = {
        "top_popular_score": pl.Float32,
        "top_popular_rank": pl.Int32,
        "global_pop_rank": pl.Int32,
        "als_score": pl.Float32,
        "als_rank": pl.Int32,
        "in_top_popular": pl.Int8,
        "in_als": pl.Int8,
    }
    for col, dtype in defaults.items():
        if col not in cols:
            fill = 0 if col.startswith("in_") else None
            exprs.append(pl.lit(fill, dtype=dtype).alias(col))

    lf = lf.with_columns(exprs) if exprs else lf
    cols = set(lazy_columns(lf))

    in_cols = [c for c in cols if c.startswith("in_") and not c.startswith("infinite")]
    rank_cols = [c for c in cols if c.endswith("_rank") or c in {"rank", "candidate_rank"}]
    score_cols = [c for c in cols if c.endswith("_score") or c in {"score", "max_source_score", "best_score"}]

    exprs = []
    if "sources_count" not in cols:
        if in_cols:
            exprs.append(pl.sum_horizontal(*[pl.col(c).fill_null(0).cast(pl.Int16) for c in in_cols]).cast(pl.Int16).alias("sources_count"))
        else:
            exprs.append(pl.lit(1, dtype=pl.Int16).alias("sources_count"))

    if "min_source_rank" not in cols:
        usable = [c for c in rank_cols if c != "candidate_rank"] or ["candidate_rank"]
        exprs.append(pl.min_horizontal(*[pl.col(c).fill_null(1_000_000_000).cast(pl.Int32, strict=False) for c in usable]).cast(pl.Int32).alias("min_source_rank"))

    if "max_source_score" not in cols:
        usable = [c for c in score_cols if c != "max_source_score"]
        if usable:
            exprs.append(pl.max_horizontal(*[pl.col(c).fill_null(-1e9).cast(pl.Float32, strict=False) for c in usable]).cast(pl.Float32).alias("max_source_score"))
        else:
            exprs.append(pl.lit(0.0, dtype=pl.Float32).alias("max_source_score"))

    return lf.with_columns(exprs) if exprs else lf


def add_candidate_features(lf: pl.LazyFrame) -> pl.LazyFrame:
    cols = set(lazy_columns(lf))
    exprs: list[pl.Expr] = []
    produced = set(cols)

    def add(expr: pl.Expr, name: str):
        if name not in produced:
            exprs.append(expr.alias(name))
            produced.add(name)

    # Generic source flags.
    for col in sorted(c for c in cols if c.startswith("in_") and not c.startswith("f_")):
        add(pl.col(col).fill_null(0).cast(pl.Float32), f"f_num__candidate__{col}")

    # Generic ranks and reciprocal ranks.
    for col in sorted(c for c in cols if (c.endswith("_rank") or c in {"candidate_rank", "rank", "min_source_rank", "global_pop_rank"}) and not c.startswith("f_")):
        safe = col.replace("rank", "rank")
        add(pl.col(col).fill_null(1_000_000).cast(pl.Float32, strict=False), f"f_num__candidate__{safe}")
        add(reciprocal_rank_expr(col, f"f_num__candidate__{safe}_rr"), f"f_num__candidate__{safe}_rr")
        add(pl.col(col).is_null().cast(pl.Float32), f"f_num__candidate__{safe}_missing")

    # Generic scores. Keep raw scores because DL cosine/logit scores may be negative.
    for col in sorted(c for c in cols if (c.endswith("_score") or c in {"score", "max_source_score", "best_score"}) and not c.startswith("f_")):
        add(pl.col(col).fill_null(0.0).cast(pl.Float32, strict=False), f"f_num__candidate__{col}")
        add(log1p_nonnegative_expr(col, f"f_num__candidate__{col}_log1p_nonnegative"), f"f_num__candidate__{col}_log1p_nonnegative")
        add(pl.col(col).is_null().cast(pl.Float32), f"f_num__candidate__{col}_missing")

    # Stable aliases used by earlier notebooks/reports.
    aliases = {
        "sources_count": "f_num__candidate__sources_count",
        "candidate_rank": "f_num__candidate__rank",
        "min_source_rank": "f_num__candidate__min_source_rank",
        "max_source_score": "f_num__candidate__max_source_score",
    }
    for src, dst in aliases.items():
        if src in cols:
            add(pl.col(src).fill_null(0).cast(pl.Float32, strict=False), dst)

    if exprs:
        lf = lf.with_columns(exprs)

    # Ensure all f_num candidate columns are numeric float32.
    f_cols = [c for c in lazy_columns(lf) if c.startswith("f_num__candidate__")]
    return lf.with_columns([pl.col(c).cast(pl.Float32, strict=False).fill_null(0.0).alias(c) for c in f_cols])


def maybe_filter_users(lf: pl.LazyFrame, user_ids: Sequence[int] | None) -> pl.LazyFrame:
    if user_ids is None:
        return lf
    ids = list(map(int, user_ids))
    return lf.filter(pl.col("user_id").cast(pl.Int64).is_in(ids))


def feature_lf_for_join(level: str, path: Path, shard_id: int | None) -> pl.LazyFrame | None:
    if level in set(RCFG.candidate_scoped_feature_levels) and path.is_dir() and shard_id is not None:
        shard_dir = path / f"user_shard={int(shard_id)}"
        if not path_has_parquet(shard_dir):
            return None
        return scan_parquet(shard_dir, hive_partitioning=True)

    if not path_has_parquet(path) and not path.exists():
        return None
    return scan_parquet(path, hive_partitioning=path.is_dir())


def join_feature_paths(lf: pl.LazyFrame, feature_paths: dict[str, Path], shard_id: int | None = None) -> pl.LazyFrame:
    catalog = item_catalog_lf()
    lf = lf.join(catalog, on="item_id", how="left")
    lf = lf.with_columns(pl.col("catalogid").fill_null(-1).cast(pl.Int64).alias("catalogid"))

    for level in RCFG.feature_levels:
        keys = LEVEL_KEYS[level]
        path = feature_paths.get(level)
        if path is None:
            continue

        feat = feature_lf_for_join(level, path, shard_id)
        if feat is None:
            continue

        casts = []
        for key in keys:
            if key in {"user_id", "item_id", "catalogid"}:
                casts.append(pl.col(key).cast(pl.Int64, strict=False).alias(key))
        if casts:
            feat = feat.with_columns(casts)

        feat_cols = [c for c in lazy_columns(feat) if c.startswith("f_")]
        if feat_cols:
            lf = lf.join(feat.select(keys + feat_cols), on=keys, how="left")

    return lf


def maybe_downsample_fit_rows(lf: pl.LazyFrame, *, downsample: bool) -> pl.LazyFrame:
    if not downsample:
        return lf

    lf = lf.with_columns(
        pl.struct(["user_id", "item_id"]).hash(seed=RCFG.random_state).alias("__rand")
    ).with_columns(
        pl.col("__rand").rank(method="ordinal").over("user_id").alias("__rand_rank")
    )

    keep_expr = (
        (pl.col("label") > 0)
        | (pl.col("candidate_rank").fill_null(1_000_000) <= int(RCFG.always_keep_top_n_candidates))
        | (pl.col("__rand_rank") <= int(RCFG.max_random_negatives_per_user))
    )
    lf = lf.filter(keep_expr).drop(["__rand", "__rand_rank"])

    if RCFG.filter_fit_groups_without_positive:
        lf = (
            lf.with_columns(pl.max("label").over("user_id").alias("__group_max_label"))
            .filter(pl.col("__group_max_label") > 0)
            .drop("__group_max_label")
        )
    return lf


def shard_ids_with_parquet(root: Path) -> set[int]:
    if not root.exists():
        return set()
    ids: set[int] = set()
    for p in root.glob("user_shard=*"):
        if not p.is_dir() or not path_has_parquet(p):
            continue
        try:
            ids.add(int(p.name.split("=", 1)[1]))
        except ValueError:
            continue
    return ids


def dataset_label_stats(dataset_dir: Path) -> dict[str, int]:
    if not path_has_parquet(dataset_dir):
        return {"rows": 0, "positives": 0}
    df = collect_streaming(
        scan_parquet(dataset_dir, hive_partitioning=True)
        .select(
            pl.len().alias("rows"),
            pl.col("label").fill_null(0).sum().alias("positives"),
        )
    )
    row = df.row(0, named=True)
    return {"rows": int(row["rows"] or 0), "positives": int(row["positives"] or 0)}


def dataset_is_reusable(split_name: str, candidates_dir: Path, out_dir: Path, *, has_labels: bool) -> bool:
    if RCFG.rebuild_datasets or not path_has_parquet(out_dir):
        return False

    expected_shards = shard_ids_with_parquet(candidates_dir)
    actual_shards = shard_ids_with_parquet(out_dir)
    missing = sorted(expected_shards - actual_shards)

    if missing:
        preview = ", ".join(map(str, missing[:12]))
        suffix = "..." if len(missing) > 12 else ""
        print(
            f"{split_name}: existing dataset is partial "
            f"({len(actual_shards)}/{len(expected_shards)} shards, missing: {preview}{suffix}) -> rebuild"
        )
        return False

    if has_labels:
        stats = dataset_label_stats(out_dir)
        if stats["positives"] <= 0:
            print(
                f"{split_name}: existing dataset has no positives "
                f"(rows={stats['rows']:,}, positives={stats['positives']:,}) -> rebuild"
            )
            return False

    print(f"{split_name}: reusable dataset exists -> {rel(out_dir)}")
    return True


def build_ranker_dataset(
    split_name: str,
    *,
    candidates_dir: Path,
    feature_paths: dict[str, Path],
    target_start: date,
    target_end: date,
    out_dir: Path,
    has_labels: bool,
    downsample_for_fit: bool,
    user_ids: Sequence[int] | None = None,
) -> Path:
    if dataset_is_reusable(split_name, candidates_dir, out_dir, has_labels=has_labels):
        return out_dir

    clean_output_dir(out_dir)
    labels_path = build_labels_for_target(split_name, target_start, target_end, has_labels=has_labels)
    labels_lf = scan_parquet(labels_path, hive_partitioning=False)

    total_rows = 0
    total_pos = 0
    total_users = 0

    for shard_id in trange(RCFG.n_shards, desc=f"Build dataset {split_name}"):
        cand_lf = scan_candidate_shard_or_none(candidates_dir, shard_id)
        if cand_lf is None:
            continue

        lf = ensure_candidate_schema(cand_lf)
        lf = maybe_filter_users(lf, user_ids)
        lf = add_candidate_features(lf)

        lf = lf.join(labels_lf, on=["user_id", "item_id"], how="left")
        lf = lf.with_columns(
            pl.col("label").fill_null(0).cast(pl.UInt8),
            pl.col("target_order_cnt").fill_null(0).cast(pl.UInt16),
            pl.col("target_delivered_order_cnt").fill_null(0).cast(pl.UInt16),
            pl.col("target_canceled_order_cnt").fill_null(0).cast(pl.UInt16),
        )

        lf = join_feature_paths(lf, feature_paths, shard_id=shard_id)
        lf = maybe_downsample_fit_rows(lf, downsample=downsample_for_fit)

        df = collect_streaming(lf)
        if df.is_empty():
            continue

        total_rows += df.height
        total_pos += int(df["label"].sum()) if "label" in df.columns else 0
        total_users += df["user_id"].n_unique()

        write_single_shard(df, out_dir, shard_id, "dataset.parquet")
        del df
        gc.collect()

    print(
        f"{split_name}: rows={total_rows:,}, positives={total_pos:,}, "
        f"users_by_shard_sum={total_users:,} -> {rel(out_dir)}"
    )
    return out_dir

## Build feature paths and datasets

In [9]:
train_feature_paths = build_feature_store_for_split(
    "train",
    RCFG.train_context_start,
    RCFG.train_context_end,
    train_candidates_dir,
)
val_feature_paths = build_feature_store_for_split(
    "val",
    RCFG.val_context_start,
    RCFG.val_context_end,
    val_candidates_dir,
)
test_feature_paths = build_feature_store_for_split(
    "test",
    RCFG.full_context_start,
    RCFG.full_context_end,
    test_candidates_dir,
)

print("train feature levels:", sorted(train_feature_paths))
print("val feature levels:  ", sorted(val_feature_paths))
print("test feature levels: ", sorted(test_feature_paths))

train_fit_dir = build_ranker_dataset(
    "train_fit",
    candidates_dir=train_candidates_dir,
    feature_paths=train_feature_paths,
    target_start=RCFG.train_target_start,
    target_end=RCFG.train_target_end,
    out_dir=DATASET_DIR / "train_fit",
    has_labels=True,
    downsample_for_fit=True,
    user_ids=train_users if RCFG.sample_train_users is not None else None,
)

val_fit_dir = build_ranker_dataset(
    "val_fit",
    candidates_dir=val_candidates_dir,
    feature_paths=val_feature_paths,
    target_start=RCFG.val_target_start,
    target_end=RCFG.val_target_end,
    out_dir=DATASET_DIR / "val_fit",
    has_labels=True,
    downsample_for_fit=True,
    user_ids=val_users if RCFG.sample_val_users is not None else None,
)

val_eval_dir = build_ranker_dataset(
    "val_eval",
    candidates_dir=val_candidates_dir,
    feature_paths=val_feature_paths,
    target_start=RCFG.val_target_start,
    target_end=RCFG.val_target_end,
    out_dir=DATASET_DIR / "val_eval",
    has_labels=True,
    downsample_for_fit=False,
    user_ids=val_users if RCFG.sample_val_users is not None else None,
)

test_dataset_dir = build_ranker_dataset(
    "test",
    candidates_dir=test_candidates_dir,
    feature_paths=test_feature_paths,
    target_start=RCFG.test_target_start,
    target_end=RCFG.test_target_end,
    out_dir=DATASET_DIR / "test",
    has_labels=RCFG.test_has_labels,
    downsample_for_fit=False,
    user_ids=test_users if RCFG.sample_test_users is not None else None,
)

train feature levels: ['category', 'item', 'user', 'user_category', 'user_item']
val feature levels:   ['category', 'item', 'user', 'user_category', 'user_item']
test feature levels:  ['category', 'item', 'user', 'user_category', 'user_item']


Build dataset train_fit:   0%|          | 0/128 [00:00<?, ?it/s]

train_fit: rows=19,309,829, positives=181,012, users_by_shard_sum=116,033 -> data\processed\reranker\final_reranker_lgbm_dl__merged_with_dl_all\dataset\train_fit


Build dataset val_fit:   0%|          | 0/128 [00:00<?, ?it/s]

val_fit: rows=17,270,419, positives=158,262, users_by_shard_sum=103,817 -> data\processed\reranker\final_reranker_lgbm_dl__merged_with_dl_all\dataset\val_fit
val_eval: reusable dataset exists -> data\processed\reranker\final_reranker_lgbm_dl__merged_with_dl_all\dataset\val_eval
test: reusable dataset exists -> data\processed\reranker\final_reranker_lgbm_dl__merged_with_dl_all\dataset\test


In [10]:
def dataset_overview(dataset_dir: Path) -> dict:
    lf = scan_parquet(dataset_dir, hive_partitioning=True)
    df = collect_streaming(
        lf.select(
            pl.len().alias("rows"),
            pl.col("user_id").n_unique().alias("users"),
            pl.col("item_id").n_unique().alias("items"),
            pl.col("label").sum().alias("positives"),
        )
    )
    row = df.row(0, named=True)
    row["positive_rate"] = float(row["positives"] / row["rows"]) if row["rows"] else 0.0
    return row


overview = pd.DataFrame([
    {"split": "train_fit", **dataset_overview(train_fit_dir)},
    {"split": "val_fit", **dataset_overview(val_fit_dir)},
    {"split": "val_eval", **dataset_overview(val_eval_dir)},
    {"split": "test", **dataset_overview(test_dataset_dir)},
])
overview_path = REPORTS_DIR / "final_reranker_dataset_overview.parquet"
overview.to_parquet(overview_path, index=False)
display(overview)
print("saved:", rel(overview_path))

,split,rows,users,items,positives,positive_rate
0,train_fit,19309829,116033,2063721,181012,0.009374
1,val_fit,17270419,103817,1950818,158262,0.009164
2,val_eval,450412159,359928,3433606,158262,0.000351
3,test,436438468,346759,3385201,148697,0.000341


saved: artifacts\reports\final_reranker_lgbm_dl__merged_with_dl_all\final_reranker_dataset_overview.parquet


In [11]:
def available_score_cols(dataset_dir: Path) -> list[str]:
    cols = set(lazy_columns(scan_parquet(dataset_dir, hive_partitioning=True)))
    candidates = [
        ("candidate_rank", False),
        ("top_popular_rank", False),
        ("als_rank", False),
        ("min_source_rank", False),
        ("max_source_score", True),
        ("dl_dssm_final_rank", False),
        ("dl_bert4rec_small_rank", False),
        ("dl_sasrec_v3_rank", False),
        ("dl_dssm_final_score", True),
        ("dl_bert4rec_small_score", True),
        ("dl_sasrec_v3_score", True),
    ]
    return [(c, higher) for c, higher in candidates if c in cols]


baseline_rows = []
for score_col, higher_is_better in available_score_cols(val_eval_dir):
    baseline_rows.append(
        evaluate_dataset_score_col(
            val_eval_dir,
            val_gt,
            score_col=score_col,
            label=score_col,
            higher_is_better=higher_is_better,
        )
    )

baseline_metrics = pd.concat(baseline_rows, ignore_index=True) if baseline_rows else pd.DataFrame()
baseline_metrics_path = REPORTS_DIR / "final_reranker_baseline_val_metrics.parquet"
baseline_metrics.to_parquet(baseline_metrics_path, index=False)
display(baseline_metrics.sort_values(["model", "k"]))
print("saved:", rel(baseline_metrics_path))

Evaluate candidate_rank:   0%|          | 0/128 [00:00<?, ?it/s]

Evaluate top_popular_rank:   0%|          | 0/128 [00:00<?, ?it/s]

Evaluate als_rank:   0%|          | 0/128 [00:00<?, ?it/s]

Evaluate min_source_rank:   0%|          | 0/128 [00:00<?, ?it/s]

Evaluate max_source_score:   0%|          | 0/128 [00:00<?, ?it/s]

Evaluate dl_dssm_final_rank:   0%|          | 0/128 [00:00<?, ?it/s]

... (3 outputs left out)

Evaluate dl_bert4rec_small_score:   0%|          | 0/128 [00:00<?, ?it/s]

Evaluate dl_sasrec_v3_score:   0%|          | 0/128 [00:00<?, ?it/s]

,model,k,users_total,users_with_predictions,user_coverage,hit_rate,recall,precision,ndcg,map,mrr
8,als_rank,20,359928,359928,1.0,0.005759,0.001496,0.000292,0.000750,0.000271,0.001010
9,als_rank,50,359928,359928,1.0,0.014067,0.003845,0.000290,0.001374,0.000342,0.001261
10,als_rank,100,359928,359928,1.0,0.027644,0.007678,0.000292,0.002240,0.000397,0.001449
11,als_rank,200,359928,359928,1.0,0.053413,0.015607,0.000295,0.003774,0.000455,0.001627
0,candidate_rank,20,359928,359928,1.0,0.030059,0.012853,0.001608,0.006312,0.003013,0.006905
1,candidate_rank,50,359928,359928,1.0,0.061157,0.023976,0.001372,0.009198,0.003376,0.007839
2,candidate_rank,100,359928,359928,1.0,0.110186,0.042449,0.001315,0.013222,0.003664,0.008524
3,candidate_rank,200,359928,359928,1.0,0.164541,0.064736,0.001056,0.017515,0.003857,0.008917
24,dl_bert4rec_small_rank,20,359928,359928,1.0,0.007921,0.002178,0.000401,0.001060,0.000380,0.001399
25,dl_bert4rec_small_rank,50,359928,359928,1.0,0.019573,0.005762,0.000404,0.001992,0.000489,0.001749


saved: artifacts\reports\final_reranker_lgbm_dl__merged_with_dl_all\final_reranker_baseline_val_metrics.parquet


## Feature selection and model matrices

In [12]:
def list_feature_cols(dataset_dir: Path) -> list[str]:
    lf = scan_parquet(dataset_dir, hive_partitioning=True)
    schema = lf.collect_schema()
    return sorted([
        col for col, dtype in zip(schema.names(), schema.dtypes())
        if col.startswith("f_") and dtype in _NUMERIC_DTYPES
    ])


def list_existing_cols(dataset_dir: Path) -> list[str]:
    lf = scan_parquet(dataset_dir, hive_partitioning=True)
    return list(lf.collect_schema().names())


def clean_feature_expr(col: str) -> pl.Expr:
    x = pl.col(col).cast(pl.Float32, strict=False)
    return (
        pl.when(x.is_null() | x.is_nan() | x.is_infinite())
        .then(pl.lit(0.0))
        .otherwise(x)
        .cast(pl.Float32)
        .alias(col)
    )


def clean_categorical_expr(col: str) -> pl.Expr:
    return pl.col(col).fill_null(-1).cast(pl.Int64, strict=False).fill_null(-1).alias(col)


def _feature_window(col: str) -> int:
    m = re.search(r"_(\d+)d(?:$|_)", col)
    return int(m.group(1)) if m else 0


def _feature_metric_priority(col: str) -> int:
    # Lower is better. Keep source/rank/click/order/candidate signals first.
    tokens = [
        ("candidate__", 0),
        ("sources_count", 1),
        ("_rank", 2),
        ("_score", 3),
        ("has_orders", 4),
        ("cnt_orders", 5),
        ("cnt_delivered_orders", 6),
        ("has_events", 7),
        ("cnt_to_cart", 8),
        ("cnt_favorite", 9),
        ("cnt_page_view", 10),
        ("days_since_last_order", 11),
        ("days_since_last_event", 12),
        ("delivered_share", 13),
        ("to_cart_per_page_view", 14),
        ("favorite_per_page_view", 15),
        ("description_per_page_view", 16),
        ("review_per_page_view", 17),
        ("cnt_events", 18),
    ]
    for token, pr in tokens:
        if token in col:
            return pr
    return 50


def _feature_level_priority(col: str) -> int:
    # Lower is better. Candidate/source features are mandatory-like; user_item is useful but can explode feature count.
    if "__candidate__" in col:
        return 0
    if "__user__" in col:
        return 10
    if "__item__" in col:
        return 20
    if "__user_item__" in col:
        return 30
    if "__user_category__" in col:
        return 40
    if "__category__" in col:
        return 50
    return 60


def feature_priority_key(col: str) -> tuple[int, int, int, str]:
    window = _feature_window(col)
    # prefer configured/recent windows; unknown window goes after known windows
    if window in RCFG.model_keep_feature_windows:
        try:
            window_pr = list(RCFG.model_keep_feature_windows).index(window)
        except ValueError:
            window_pr = 99
    else:
        window_pr = 99
    return (_feature_level_priority(col), _feature_metric_priority(col), window_pr, col)


def select_memory_safe_numeric_features(all_numeric_features: list[str]) -> list[str]:
    cols = [c for c in all_numeric_features if (_feature_window(c) == 0 or _feature_window(c) in RCFG.model_keep_feature_windows)]

    # Candidate features encode source ranks/scores and should be kept even under aggressive caps.
    candidate_cols = sorted([c for c in cols if "__candidate__" in c], key=feature_priority_key)
    other_cols = sorted([c for c in cols if "__candidate__" not in c], key=feature_priority_key)

    if RCFG.model_max_numeric_features is None:
        selected = candidate_cols + other_cols
    else:
        max_n = int(RCFG.model_max_numeric_features)
        selected = (candidate_cols + other_cols)[:max_n]

    missing_candidate = [c for c in candidate_cols if c not in selected]
    if missing_candidate:
        # If candidate feature count alone exceeds cap, we still respect cap but warn.
        print(f"WARNING: candidate feature count exceeds cap; dropped {len(missing_candidate)} candidate features")

    return selected


all_numeric_feature_cols = sorted(set(list_feature_cols(train_fit_dir)) & set(list_feature_cols(val_fit_dir)))
numeric_feature_cols = select_memory_safe_numeric_features(all_numeric_feature_cols)
all_train_cols = set(list_existing_cols(train_fit_dir))

base_catboost_cat_cols = ["catalogid"]
if RCFG.catboost_use_id_features:
    base_catboost_cat_cols = ["user_id", "item_id", "catalogid"]
catboost_cat_cols = [c for c in base_catboost_cat_cols if c in all_train_cols or c in {"user_id", "item_id"}]
catboost_feature_cols = catboost_cat_cols + numeric_feature_cols

print(f"numeric features available: {len(all_numeric_feature_cols):,}")
print(f"numeric features selected for model: {len(numeric_feature_cols):,}")
print(f"CatBoost categorical features: {catboost_cat_cols}")
print("Selected feature preview:")
for c in numeric_feature_cols[:80]:
    print(" ", c)

feature_list_path = MODELS_DIR / "feature_cols.json"
feature_manifest = {
    "all_numeric_feature_cols_count": len(all_numeric_feature_cols),
    "numeric_feature_cols": numeric_feature_cols,
    "catboost_cat_cols": catboost_cat_cols,
    "catboost_feature_cols": catboost_feature_cols,
    "selection_config": {
        "model_max_numeric_features": RCFG.model_max_numeric_features,
        "model_keep_feature_windows": list(RCFG.model_keep_feature_windows),
        "model_train_max_rows": RCFG.model_train_max_rows,
        "model_val_max_rows": RCFG.model_val_max_rows,
    },
}
feature_list_path.write_text(json.dumps(feature_manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved:", rel(feature_list_path))


numeric features available: 566
numeric features selected for model: 180
CatBoost categorical features: ['catalogid']
Selected feature preview:
  f_num__candidate__als_rank
  f_num__candidate__als_rank_missing
  f_num__candidate__als_rank_rr
  f_num__candidate__als_score
  f_num__candidate__als_score_log1p_nonnegative
  f_num__candidate__als_score_missing
  f_num__candidate__candidate_rank
  f_num__candidate__candidate_rank_missing
  f_num__candidate__candidate_rank_rr
  f_num__candidate__clip_faiss_rank
  f_num__candidate__clip_faiss_rank_missing
  f_num__candidate__clip_faiss_rank_rr
  f_num__candidate__clip_faiss_score
  f_num__candidate__clip_faiss_score_log1p_nonnegative
  f_num__candidate__clip_faiss_score_missing
  f_num__candidate__clip_hnsw_rank
  f_num__candidate__clip_hnsw_rank_missing
  f_num__candidate__clip_hnsw_rank_rr
  f_num__candidate__clip_hnsw_score
  f_num__candidate__clip_hnsw_score_log1p_nonnegative
  f_num__candidate__clip_hnsw_score_missing
  f_num__candidate__

In [13]:
def dataset_basic_stats(dataset_dir: Path) -> dict[str, int]:
    lf = scan_parquet(dataset_dir, hive_partitioning=True)
    df = collect_streaming(
        lf.select(
            pl.len().alias("rows"),
            pl.col("label").fill_null(0).sum().alias("positives"),
            pl.col("user_id").n_unique().alias("users"),
        )
    )
    row = df.row(0, named=True)
    return {k: int(row[k] or 0) for k in ["rows", "positives", "users"]}


def add_fit_row_sampling(
    lf: pl.LazyFrame,
    *,
    total_rows: int,
    total_positives: int,
    max_rows: int | None,
    split_name: str,
) -> pl.LazyFrame:
    """Sample negatives before pandas conversion while always keeping positives by default."""
    if max_rows is None or total_rows <= int(max_rows):
        print(f"{split_name}: no model-row sampling ({total_rows:,} <= {max_rows})")
        return lf

    max_rows = int(max_rows)
    pos = int(total_positives)
    neg = max(total_rows - pos, 0)

    if RCFG.model_force_all_positives:
        neg_budget = max(max_rows - pos, 1)
        if neg <= 0:
            print(f"{split_name}: only positives; no negative sampling")
            return lf.filter(pl.col("label") > 0)
        threshold = max(1, min(RCFG.model_fit_hash_mod, int(RCFG.model_fit_hash_mod * (neg_budget / neg))))
        keep_expr = (
            (pl.col("label") > 0)
            | ((pl.struct(["user_id", "item_id"]).hash(seed=RCFG.random_state) % int(RCFG.model_fit_hash_mod)) < threshold)
        )
        expected = pos + int(neg * threshold / RCFG.model_fit_hash_mod)
        print(
            f"{split_name}: model-row sampling rows={total_rows:,}, positives={pos:,}, "
            f"max_rows={max_rows:,}, neg_threshold={threshold}/{RCFG.model_fit_hash_mod}, expected≈{expected:,}"
        )
        return lf.filter(keep_expr)

    threshold = max(1, min(RCFG.model_fit_hash_mod, int(RCFG.model_fit_hash_mod * (max_rows / total_rows))))
    print(f"{split_name}: row sampling threshold={threshold}/{RCFG.model_fit_hash_mod}")
    return lf.filter((pl.struct(["user_id", "item_id"]).hash(seed=RCFG.random_state) % int(RCFG.model_fit_hash_mod)) < threshold)


def collect_model_pdf(
    dataset_dir: Path,
    numeric_features: list[str],
    categorical_features: list[str] | None = None,
    *,
    split_name: str,
    max_rows: int | None,
) -> pd.DataFrame:
    categorical_features = categorical_features or []
    lf = scan_parquet(dataset_dir, hive_partitioning=True)
    available = set(lazy_columns(lf))

    numeric_features = [c for c in numeric_features if c in available]
    select_cols = ["user_id", "item_id", "label"]

    for col in categorical_features:
        if col not in select_cols:
            if col in available or col in {"user_id", "item_id"}:
                select_cols.append(col)

    for col in numeric_features:
        if col not in select_cols:
            select_cols.append(col)

    stats = dataset_basic_stats(dataset_dir)
    lf = add_fit_row_sampling(
        lf,
        total_rows=stats["rows"],
        total_positives=stats["positives"],
        max_rows=max_rows,
        split_name=split_name,
    )

    missing_exprs = []
    if "catalogid" in categorical_features and "catalogid" not in available:
        missing_exprs.append(pl.lit(-1, dtype=pl.Int64).alias("catalogid"))
    if missing_exprs:
        lf = lf.with_columns(missing_exprs)

    clean_exprs = [clean_feature_expr(c) for c in numeric_features]
    clean_exprs += [clean_categorical_expr(c) for c in categorical_features if c in select_cols]

    # Select only the capped feature list before collect; this is the core memory fix.
    df = collect_streaming(lf.with_columns(clean_exprs).select(select_cols))
    actual_rows = df.height
    approx_numeric_gb = actual_rows * max(len(numeric_features), 1) * 4 / (1024 ** 3)
    print(
        f"{split_name}: collected polars rows={actual_rows:,}, cols={len(select_cols):,}, "
        f"numeric_block≈{approx_numeric_gb:.2f} GB before pandas overhead"
    )

    pdf = df.to_pandas()
    del df
    gc.collect()

    pdf["user_id"] = pdf["user_id"].astype(np.int64)
    pdf["item_id"] = pdf["item_id"].astype(np.int64)
    pdf["label"] = pdf["label"].astype(np.int32)

    for col in numeric_features:
        if col in pdf.columns:
            pdf[col] = pdf[col].astype(np.float32, copy=False)

    for col in categorical_features:
        if col in pdf.columns:
            pdf[col] = pdf[col].fillna(-1).astype(np.int64)

    return pdf


def prepare_grouped_pdf(pdf: pd.DataFrame, *, split_name: str) -> tuple[pd.DataFrame, np.ndarray]:
    before_rows = len(pdf)
    before_users = pdf["user_id"].nunique()

    if RCFG.filter_fit_groups_without_positive:
        keep = pdf.groupby("user_id")["label"].transform("max") > 0
        pdf = pdf.loc[keep].copy()

    pdf = pdf.sort_values(["user_id", "item_id"], kind="mergesort").reset_index(drop=True)
    group = pdf.groupby("user_id", sort=False).size().to_numpy(dtype=np.int32)

    print(
        f"{split_name}: grouped rows {before_rows:,}->{len(pdf):,}, "
        f"users {before_users:,}->{len(group):,}, positives={int(pdf['label'].sum()):,}"
    )
    return pdf, group


train_pdf = collect_model_pdf(
    train_fit_dir,
    numeric_feature_cols,
    catboost_cat_cols,
    split_name="train_fit",
    max_rows=RCFG.model_train_max_rows,
)
val_pdf = collect_model_pdf(
    val_fit_dir,
    numeric_feature_cols,
    catboost_cat_cols,
    split_name="val_fit",
    max_rows=RCFG.model_val_max_rows,
)

train_pdf, train_group = prepare_grouped_pdf(train_pdf, split_name="train_fit")
val_pdf, val_group = prepare_grouped_pdf(val_pdf, split_name="val_fit")

print("train_pdf:", train_pdf.shape, "groups:", len(train_group), "positives:", int(train_pdf["label"].sum()))
print("val_pdf:  ", val_pdf.shape, "groups:", len(val_group), "positives:", int(val_pdf["label"].sum()))


train_fit: model-row sampling rows=19,309,829, positives=181,012, max_rows=4,000,000, neg_threshold=199646/1000003, expected≈3,999,992
train_fit: collected polars rows=3,999,698, cols=184, numeric_block≈2.68 GB before pandas overhead


C:\Users\denis\AppData\Local\Temp\ipykernel_16760\3670414102.py:114: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  pdf[col] = pdf[col].astype(np.float32, copy=False)
C:\Users\denis\AppData\Local\Temp\ipykernel_16760\3670414102.py:114: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  pdf[col] = pdf[col].astype(np.float32, copy=False)
C:\Users\denis\AppData\Local\Temp\ipykernel_16760\3670414102.py:114: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers 

val_fit: model-row sampling rows=17,270,419, positives=158,262, max_rows=1,200,000, neg_threshold=60877/1000003, expected≈1,199,995
val_fit: collected polars rows=1,199,793, cols=184, numeric_block≈0.80 GB before pandas overhead


C:\Users\denis\AppData\Local\Temp\ipykernel_16760\3670414102.py:114: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  pdf[col] = pdf[col].astype(np.float32, copy=False)
C:\Users\denis\AppData\Local\Temp\ipykernel_16760\3670414102.py:114: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  pdf[col] = pdf[col].astype(np.float32, copy=False)
C:\Users\denis\AppData\Local\Temp\ipykernel_16760\3670414102.py:114: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers 

train_fit: grouped rows 3,999,698->3,999,698, users 116,033->116,033, positives=181,012
val_fit: grouped rows 1,199,793->1,199,793, users 103,817->103,817, positives=158,262
train_pdf: (3999698, 184) groups: 116033 positives: 181012
val_pdf:   (1199793, 184) groups: 103817 positives: 158262


## Ranking model selection

In [14]:
def parse_metric_name(metric_name: str) -> tuple[str, int]:
    name, k = metric_name.lower().split("@")
    return name, int(k)


def metric_from_metrics_df(metrics: pd.DataFrame, metric_name: str) -> float:
    name, k = parse_metric_name(metric_name)
    return float(metrics.loc[metrics["k"] == k, name].iloc[0])


def evaluate_scored_pandas(
    pdf: pd.DataFrame,
    ground_truth: dict[int, list[int]],
    *,
    score_col: str = "score",
    label: str = "model",
) -> pd.DataFrame:
    return evaluate_scored_df(pdf[["user_id", "item_id", score_col]], ground_truth, score_col=score_col, label=label)


def make_model_record(
    *,
    name: str,
    model_type: str,
    model,
    features: list[str],
    categorical_features: list[str],
    params: dict,
    best_iteration: int | None,
    tuning_time_sec: float,
    selection_fit_time_sec: float,
    study: optuna.Study | None,
) -> dict:
    return {
        "name": name,
        "model_type": model_type,
        "model": model,
        "features": features,
        "categorical_features": categorical_features,
        "params": params,
        "best_iteration": best_iteration,
        "tuning_time_sec": float(tuning_time_sec),
        "selection_fit_time_sec": float(selection_fit_time_sec),
        "study": study,
    }


def predict_model(record: dict, pdf: pd.DataFrame) -> np.ndarray:
    features = record["features"]
    model = record["model"]

    if record["model_type"] == "lgbm":
        num_iteration = record.get("best_iteration")
        return model.predict(pdf[features], num_iteration=num_iteration)

    if record["model_type"] == "catboost":
        cat_features = [c for c in record["categorical_features"] if c in features]
        pool = Pool(pdf[features], cat_features=cat_features)
        return model.predict(pool)

    raise ValueError(record["model_type"])

In [15]:
def base_lgbm_params() -> dict:
    return {
        "objective": "lambdarank",
        "metric": "ndcg",
        "boosting_type": "gbdt",
        "random_state": RCFG.random_state,
        "n_jobs": -1,
        "verbosity": -1,
        "label_gain": [0, 1],
        "force_col_wise": True,
    }


def fit_lgbm_ranker(
    params: dict,
    train_data: pd.DataFrame,
    train_groups: np.ndarray,
    *,
    valid_data: pd.DataFrame | None = None,
    valid_groups: np.ndarray | None = None,
):
    features = numeric_feature_cols
    model = lgbm.LGBMRanker(**base_lgbm_params(), **params)

    callbacks = []
    eval_set = None
    eval_group = None

    if valid_data is not None and valid_groups is not None:
        eval_set = [(valid_data[features], valid_data["label"])]
        eval_group = [valid_groups]
        callbacks = [
            lgbm.early_stopping(RCFG.early_stopping_rounds, verbose=False),
            lgbm.log_evaluation(period=0),
        ]

    model.fit(
        train_data[features],
        train_data["label"],
        group=train_groups,
        eval_set=eval_set,
        eval_group=eval_group,
        eval_at=list(RCFG.metric_ks),
        callbacks=callbacks,
    )
    return model


def tune_lgbm_ranker() -> dict | None:
    if not RCFG.run_lightgbm:
        print("LightGBM disabled by config")
        return None

    opt_metric_name, opt_k = parse_metric_name(RCFG.optimize_metric)

    def objective_lgbm(trial: optuna.Trial) -> float:
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 300, 1200),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.12, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 31, 191),
            "max_depth": trial.suggest_int("max_depth", 3, 14),
            "min_child_samples": trial.suggest_int("min_child_samples", 20, 350),
            "subsample": trial.suggest_float("subsample", 0.65, 1.0),
            "subsample_freq": 1,
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.65, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-5, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-5, 10.0, log=True),
            "lambdarank_truncation_level": trial.suggest_int("lambdarank_truncation_level", 50, 200),
        }

        model = fit_lgbm_ranker(params, train_pdf, train_group, valid_data=val_pdf, valid_groups=val_group)
        best_iter = getattr(model, "best_iteration_", None)
        if best_iter is None or best_iter <= 0:
            best_iter = params["n_estimators"]

        val_scores = model.predict(val_pdf[numeric_feature_cols], num_iteration=best_iter)
        scored = val_pdf[["user_id", "item_id"]].copy()
        scored["score"] = val_scores
        metrics = evaluate_scored_pandas(scored, val_gt, score_col="score", label="lgbm_trial")
        metric_value = float(metrics.loc[metrics["k"] == opt_k, opt_metric_name].iloc[0])

        trial.set_user_attr("best_iteration", int(best_iter))
        trial.set_user_attr("metrics", metrics.to_dict(orient="records"))
        return metric_value

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=RCFG.random_state), study_name="final_lgbm_ranker")

    t0 = time.perf_counter()
    study.optimize(
        objective_lgbm,
        n_trials=RCFG.optuna_trials,
        timeout=RCFG.optuna_timeout_sec,
        show_progress_bar=True,
    )
    tuning_time_sec = time.perf_counter() - t0

    best_params = study.best_params.copy()
    best_iteration = study.best_trial.user_attrs.get("best_iteration")
    if best_iteration is not None and int(best_iteration) > 0:
        best_params["n_estimators"] = int(best_iteration)

    t0 = time.perf_counter()
    model = fit_lgbm_ranker(best_params, train_pdf, train_group, valid_data=val_pdf, valid_groups=val_group)
    selection_fit_time_sec = time.perf_counter() - t0

    record = make_model_record(
        name="lgbm_ranker",
        model_type="lgbm",
        model=model,
        features=numeric_feature_cols,
        categorical_features=[],
        params=best_params,
        best_iteration=int(best_iteration) if best_iteration is not None else None,
        tuning_time_sec=tuning_time_sec,
        selection_fit_time_sec=selection_fit_time_sec,
        study=study,
    )

    trials_path = REPORTS_DIR / "final_reranker_lgbm_optuna_trials.parquet"
    study.trials_dataframe().to_parquet(trials_path, index=False)
    print(f"LightGBM best {RCFG.optimize_metric}: {study.best_value:.6f}")
    print("saved:", rel(trials_path))
    return record


lgbm_record = tune_lgbm_ranker()


  0%|          | 0/12 [00:00<?, ?it/s]

LightGBM best ndcg@100: 0.119368
saved: artifacts\reports\final_reranker_lgbm_dl__merged_with_dl_all\final_reranker_lgbm_optuna_trials.parquet


In [16]:
def make_catboost_pool(pdf: pd.DataFrame, features: list[str], categorical_features: list[str]):
    if Pool is None:
        raise ImportError("catboost is not installed")
    cat_features = [c for c in categorical_features if c in features]
    return Pool(
        data=pdf[features],
        label=pdf["label"].astype(np.float32),
        group_id=pdf["user_id"].astype(np.int64),
        cat_features=cat_features,
    )


def base_catboost_params() -> dict:
    _, opt_k = parse_metric_name(RCFG.optimize_metric)
    return {
        "loss_function": "YetiRank",
        "eval_metric": f"NDCG:top={opt_k}",
        "random_seed": RCFG.random_state,
        "thread_count": -1,
        "allow_writing_files": False,
        "verbose": False,
        "od_type": "Iter",
        "od_wait": RCFG.early_stopping_rounds,
    }


def fit_catboost_ranker(params: dict, train_data: pd.DataFrame, *, valid_data: pd.DataFrame | None = None):
    features = catboost_feature_cols
    model = CatBoostRanker(**base_catboost_params(), **params)
    train_pool = make_catboost_pool(train_data, features, catboost_cat_cols)
    eval_set = make_catboost_pool(valid_data, features, catboost_cat_cols) if valid_data is not None else None
    model.fit(train_pool, eval_set=eval_set, use_best_model=eval_set is not None, verbose=False)
    return model


def tune_catboost_ranker() -> dict | None:
    if not RCFG.run_catboost:
        print("CatBoost disabled by config")
        return None
    if not HAS_CATBOOST:
        print("CatBoost is not installed. Skipping CatBoost challenger. Install with: pip install catboost")
        return None

    opt_metric_name, opt_k = parse_metric_name(RCFG.optimize_metric)

    def objective_catboost(trial: optuna.Trial) -> float:
        params = {
            "iterations": trial.suggest_int("iterations", 400, 1200),
            "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.20, log=True),
            "depth": trial.suggest_int("depth", 4, 10),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-2, 100.0, log=True),
            "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
            "bootstrap_type": "Bayesian",
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 10.0),
            "border_count": trial.suggest_int("border_count", 32, 254),
        }

        model = fit_catboost_ranker(params, train_pdf, valid_data=val_pdf)
        best_iter = model.get_best_iteration()
        if best_iter is None or best_iter <= 0:
            best_iter = params["iterations"]

        pool = Pool(val_pdf[catboost_feature_cols], cat_features=[c for c in catboost_cat_cols if c in catboost_feature_cols])
        val_scores = model.predict(pool)
        scored = val_pdf[["user_id", "item_id"]].copy()
        scored["score"] = val_scores
        metrics = evaluate_scored_pandas(scored, val_gt, score_col="score", label="catboost_trial")
        metric_value = float(metrics.loc[metrics["k"] == opt_k, opt_metric_name].iloc[0])

        trial.set_user_attr("best_iteration", int(best_iter))
        trial.set_user_attr("metrics", metrics.to_dict(orient="records"))
        return metric_value

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=RCFG.random_state), study_name="final_catboost_ranker")

    t0 = time.perf_counter()
    study.optimize(objective_catboost, n_trials=RCFG.catboost_optuna_trials, timeout=RCFG.optuna_timeout_sec, show_progress_bar=True)
    tuning_time_sec = time.perf_counter() - t0

    best_params = study.best_params.copy()
    best_iteration = study.best_trial.user_attrs.get("best_iteration")
    if best_iteration is not None and int(best_iteration) > 0:
        best_params["iterations"] = int(best_iteration)

    t0 = time.perf_counter()
    model = fit_catboost_ranker(best_params, train_pdf, valid_data=val_pdf)
    selection_fit_time_sec = time.perf_counter() - t0

    record = make_model_record(
        name="catboost_ranker",
        model_type="catboost",
        model=model,
        features=catboost_feature_cols,
        categorical_features=catboost_cat_cols,
        params=best_params,
        best_iteration=int(best_iteration) if best_iteration is not None else None,
        tuning_time_sec=tuning_time_sec,
        selection_fit_time_sec=selection_fit_time_sec,
        study=study,
    )

    trials_path = REPORTS_DIR / "final_reranker_catboost_optuna_trials.parquet"
    study.trials_dataframe().to_parquet(trials_path, index=False)
    print(f"CatBoost best {RCFG.optimize_metric}: {study.best_value:.6f}")
    print("saved:", rel(trials_path))
    return record


catboost_record = tune_catboost_ranker()

CatBoost disabled by config


## Score validation, compare models, refit selected production model

In [17]:
def save_model_record(record: dict, suffix: str = "selection") -> dict[str, str]:
    name = record["name"]
    outputs: dict[str, str] = {}

    if record["model_type"] == "lgbm":
        joblib_path = MODELS_DIR / f"{name}_{suffix}.joblib"
        txt_path = MODELS_DIR / f"{name}_{suffix}.txt"
        joblib.dump(record["model"], joblib_path)
        record["model"].booster_.save_model(str(txt_path))
        outputs["joblib"] = rel(joblib_path)
        outputs["txt"] = rel(txt_path)

    elif record["model_type"] == "catboost":
        cbm_path = MODELS_DIR / f"{name}_{suffix}.cbm"
        joblib_path = MODELS_DIR / f"{name}_{suffix}.joblib"
        record["model"].save_model(str(cbm_path))
        joblib.dump(record["model"], joblib_path)
        outputs["cbm"] = rel(cbm_path)
        outputs["joblib"] = rel(joblib_path)
    else:
        raise ValueError(record["model_type"])
    return outputs


def score_dataset_dir_to_dir(record: dict, dataset_dir: Path, out_dir: Path, *, k: int = RCFG.final_k) -> tuple[Path, float]:
    clean_output_dir(out_dir)

    features = record["features"]
    categorical_features = set(record["categorical_features"])
    required_cols = [
        "user_id", "item_id", "label", "candidate_rank",
        "top_popular_rank", "als_rank", "min_source_rank", "max_source_score",
    ]
    select_cols = list(dict.fromkeys(required_cols + features))

    rows_written = 0
    t0 = time.perf_counter()

    files_by_shard = {}
    for shard_id in range(RCFG.n_shards):
        shard_dir = dataset_dir / f"user_shard={shard_id}"
        if path_has_parquet(shard_dir):
            files_by_shard[shard_id] = candidate_files(shard_dir)

    for shard_id in tqdm(files_by_shard, desc=f"Score {record['name']} on {rel(dataset_dir)}"):
        part_lf = scan_parquet(dataset_dir / f"user_shard={shard_id}", hive_partitioning=True)
        available = set(lazy_columns(part_lf))

        missing_exprs = []
        for col in select_cols:
            if col in available:
                continue
            if col in categorical_features or col in {"user_id", "item_id", "catalogid"}:
                missing_exprs.append(pl.lit(-1, dtype=pl.Int64).alias(col))
            elif col == "label":
                missing_exprs.append(pl.lit(0, dtype=pl.UInt8).alias(col))
            elif col.endswith("_rank") or col == "candidate_rank":
                missing_exprs.append(pl.lit(1_000_000, dtype=pl.Int32).alias(col))
            else:
                missing_exprs.append(pl.lit(0.0, dtype=pl.Float32).alias(col))
        if missing_exprs:
            part_lf = part_lf.with_columns(missing_exprs)

        clean_exprs = []
        for col in features:
            if col in categorical_features or col in {"user_id", "item_id", "catalogid"}:
                clean_exprs.append(clean_categorical_expr(col))
            else:
                clean_exprs.append(clean_feature_expr(col))

        part = collect_streaming(part_lf.with_columns(clean_exprs).select(select_cols))
        if part.is_empty():
            continue

        pdf = part.select(features).to_pandas()
        for col in categorical_features:
            if col in pdf.columns:
                pdf[col] = pdf[col].fillna(-1).astype(np.int64)

        scores = predict_model(record, pdf)

        scored = (
            part
            .with_columns(pl.Series("ranker_score", scores).cast(pl.Float32))
            .sort(["user_id", "ranker_score", "item_id"], descending=[False, True, False])
            .with_columns(pl.col("item_id").cum_count().over("user_id").cast(pl.Int32).alias("rank"))
            .filter(pl.col("rank") <= int(k))
            .select(
                "user_id", "item_id", "ranker_score", "rank", "label",
                "candidate_rank", "top_popular_rank", "als_rank", "min_source_rank", "max_source_score",
            )
        )

        rows_written += write_single_shard(scored, out_dir, shard_id, "reranked.parquet")
        del part, pdf, scored
        gc.collect()

    elapsed = time.perf_counter() - t0
    print(f"{record['name']}: scored rows={rows_written:,}, time={elapsed:.1f}s -> {rel(out_dir)}")
    return out_dir, elapsed


MODEL_COMPARISON_DIR = RERANKED_DIR / "model_comparison"
MODEL_COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

model_records = [r for r in [lgbm_record, catboost_record] if r is not None]
if not model_records:
    raise RuntimeError("No ranking models were trained. Enable LightGBM or CatBoost.")

saved_model_paths = {record["name"]: save_model_record(record, suffix="selection") for record in model_records}

comparison_metric_rows = []
comparison_summary_rows = []

for record in model_records:
    val_out_dir = MODEL_COMPARISON_DIR / record["name"] / "val"
    scored_dir, predict_time_sec = score_dataset_dir_to_dir(record, val_eval_dir, val_out_dir, k=max(RCFG.metric_ks))
    metrics = evaluate_scored_dir(scored_dir, val_gt, rank_col="rank", label=record["name"])
    comparison_metric_rows.append(metrics)

    row = {
        "model": record["name"],
        "model_type": record["model_type"],
        "features_count": len(record["features"]),
        "categorical_features": ",".join(record["categorical_features"]),
        "tuning_time_sec": record["tuning_time_sec"],
        "selection_fit_time_sec": record["selection_fit_time_sec"],
        "val_predict_time_sec": predict_time_sec,
        "best_iteration": record["best_iteration"],
        "model_paths": json.dumps(saved_model_paths[record["name"]], ensure_ascii=False),
    }
    for _, mrow in metrics.iterrows():
        k = int(mrow["k"])
        row[f"ndcg@{k}"] = float(mrow["ndcg"])
        row[f"map@{k}"] = float(mrow["map"])
        row[f"recall@{k}"] = float(mrow["recall"])
        row[f"hit_rate@{k}"] = float(mrow["hit_rate"])
    row["optimize_metric_value"] = metric_from_metrics_df(metrics, RCFG.optimize_metric)
    comparison_summary_rows.append(row)

model_val_metrics = pd.concat(comparison_metric_rows, ignore_index=True)
model_comparison = pd.DataFrame(comparison_summary_rows).sort_values(["optimize_metric_value", "val_predict_time_sec"], ascending=[False, True])

model_comparison_path = REPORTS_DIR / "final_reranker_model_comparison.parquet"
model_val_metrics_path = REPORTS_DIR / "final_reranker_model_val_metrics.parquet"
model_comparison.to_parquet(model_comparison_path, index=False)
model_val_metrics.to_parquet(model_val_metrics_path, index=False)

print("saved:", rel(model_comparison_path))
print("saved:", rel(model_val_metrics_path))
display(model_comparison)
display(model_val_metrics.sort_values(["model", "k"]))

Score lgbm_ranker on data\processed\reranker\final_reranker_lgbm_dl__merged_with_dl_all\dataset\val_eval:   0%…

lgbm_ranker: scored rows=71,985,600, time=606.9s -> data\datamart\candidates\reranked\final_reranker_lgbm_dl__merged_with_dl_all\model_comparison\lgbm_ranker\val


Evaluate lgbm_ranker:   0%|          | 0/128 [00:00<?, ?it/s]

saved: artifacts\reports\final_reranker_lgbm_dl__merged_with_dl_all\final_reranker_model_comparison.parquet
saved: artifacts\reports\final_reranker_lgbm_dl__merged_with_dl_all\final_reranker_model_val_metrics.parquet


,model,model_type,features_count,categorical_features,tuning_time_sec,selection_fit_time_sec,val_predict_time_sec,best_iteration,model_paths,ndcg@20,map@20,recall@20,hit_rate@20,ndcg@50,map@50,recall@50,hit_rate@50,ndcg@100,map@100,recall@100,hit_rate@100,ndcg@200,map@200,recall@200,hit_rate@200,optimize_metric_value
0,lgbm_ranker,lgbm,180,,956.912602,10.6015,606.906563,53,"{""joblib"": ""models\\reranker\\final_reranker_l...",0.016838,0.009578,0.026519,0.073051,0.021144,0.010151,0.042339,0.117815,0.025054,0.010449,0.05934,0.161291,0.029363,0.010652,0.081128,0.210617,0.025054


,model,k,users_total,users_with_predictions,user_coverage,hit_rate,recall,precision,ndcg,map,mrr
0,lgbm_ranker,20,359928,359928,1.0,0.073051,0.026519,0.004201,0.016838,0.009578,0.022917
1,lgbm_ranker,50,359928,359928,1.0,0.117815,0.042339,0.002923,0.021144,0.010151,0.024322
2,lgbm_ranker,100,359928,359928,1.0,0.161291,0.059340,0.002147,0.025054,0.010449,0.024938
3,lgbm_ranker,200,359928,359928,1.0,0.210617,0.081128,0.001501,0.029363,0.010652,0.025291


In [18]:
best_model_name = str(model_comparison.iloc[0]["model"])
best_selection_record = next(r for r in model_records if r["name"] == best_model_name)
print(f"Best model by {RCFG.optimize_metric}: {best_model_name}")


def fit_production_model_from_record(record: dict) -> dict:
    """
    By default we reuse the selected model trained on sampled train_fit with val_fit early stopping.
    This is intentional for the memory-safe notebook: refitting on train+val requires an extra large
    pandas concat and can trigger another RAM spike.

    Set RCFG.refit_production_on_train_val=True only if there is enough RAM.
    """
    if not RCFG.refit_production_on_train_val:
        prod_record = record.copy()
        prod_record["production_fit_time_sec"] = 0.0
        prod_record["production_refit"] = False
        print("Production refit disabled: using best selection model for val/test scoring.")
        return prod_record

    train_val_pdf = pd.concat([train_pdf, val_pdf], ignore_index=True)
    train_val_pdf, train_val_group = prepare_grouped_pdf(train_val_pdf, split_name="train_val_production")

    params = record["params"].copy()
    if record["best_iteration"] is not None:
        if record["model_type"] == "lgbm":
            params["n_estimators"] = int(record["best_iteration"])
        elif record["model_type"] == "catboost":
            params["iterations"] = int(record["best_iteration"])

    t0 = time.perf_counter()
    if record["model_type"] == "lgbm":
        model = fit_lgbm_ranker(params, train_val_pdf, train_val_group)
    elif record["model_type"] == "catboost":
        model = fit_catboost_ranker(params, train_val_pdf, valid_data=None)
    else:
        raise ValueError(record["model_type"])
    production_fit_time_sec = time.perf_counter() - t0

    prod_record = record.copy()
    prod_record["model"] = model
    prod_record["params"] = params
    prod_record["production_fit_time_sec"] = production_fit_time_sec
    prod_record["production_refit"] = True

    del train_val_pdf, train_val_group
    gc.collect()
    return prod_record


production_record = fit_production_model_from_record(best_selection_record)
production_model_paths = save_model_record(production_record, suffix="production")

best_joblib_path = MODELS_DIR / "best_ranker.joblib"
joblib.dump(production_record["model"], best_joblib_path)
production_model_paths["best_ranker_joblib"] = rel(best_joblib_path)

if production_record["model_type"] == "lgbm":
    best_txt_path = MODELS_DIR / "best_ranker.txt"
    production_record["model"].booster_.save_model(str(best_txt_path))
    production_model_paths["best_ranker_txt"] = rel(best_txt_path)
elif production_record["model_type"] == "catboost":
    best_cbm_path = MODELS_DIR / "best_ranker.cbm"
    production_record["model"].save_model(str(best_cbm_path))
    production_model_paths["best_ranker_cbm"] = rel(best_cbm_path)

print("production model paths:")
print(json.dumps(production_model_paths, ensure_ascii=False, indent=2))
print(f"production fit time: {production_record['production_fit_time_sec']:.1f}s")


Best model by ndcg@100: lgbm_ranker
Production refit disabled: using best selection model for val/test scoring.
production model paths:
{
  "joblib": "models\\reranker\\final_reranker_lgbm_dl__merged_with_dl_all\\lgbm_ranker_production.joblib",
  "txt": "models\\reranker\\final_reranker_lgbm_dl__merged_with_dl_all\\lgbm_ranker_production.txt",
  "best_ranker_joblib": "models\\reranker\\final_reranker_lgbm_dl__merged_with_dl_all\\best_ranker.joblib",
  "best_ranker_txt": "models\\reranker\\final_reranker_lgbm_dl__merged_with_dl_all\\best_ranker.txt"
}
production fit time: 0.0s


In [19]:
def get_feature_importance(record: dict) -> pd.DataFrame:
    features = record["features"]
    model = record["model"]

    if record["model_type"] == "lgbm":
        return pd.DataFrame({
            "feature": features,
            "importance_gain": model.booster_.feature_importance(importance_type="gain"),
            "importance_split": model.booster_.feature_importance(importance_type="split"),
        }).sort_values("importance_gain", ascending=False)

    if record["model_type"] == "catboost":
        pool = Pool(
            train_pdf[features],
            label=train_pdf["label"],
            group_id=train_pdf["user_id"],
            cat_features=[c for c in record["categorical_features"] if c in features],
        )
        importance = model.get_feature_importance(pool, type="FeatureImportance")
        return pd.DataFrame({"feature": features, "importance": importance}).sort_values("importance", ascending=False)

    raise ValueError(record["model_type"])


importance = get_feature_importance(production_record)
importance_path = REPORTS_DIR / "final_reranker_feature_importance.parquet"
importance.to_parquet(importance_path, index=False)
display(importance.head(50))
print("saved:", rel(importance_path))

,feature,importance_gain,importance_split
54,f_num__candidate__i2i_bm25_score,323799.488817,339
6,f_num__candidate__candidate_rank,130356.412301,571
15,f_num__candidate__clip_hnsw_rank,117822.750261,228
8,f_num__candidate__candidate_rank_rr,107908.191088,311
55,f_num__candidate__i2i_bm25_score_log1p_nonnega...,107461.280681,180
2,f_num__candidate__als_rank_rr,97134.331371,109
53,f_num__candidate__i2i_bm25_rank_rr,75393.340089,285
30,f_num__candidate__dl_dssm_final_pop_prior,72788.696748,227
0,f_num__candidate__als_rank,69399.703131,270
3,f_num__candidate__als_score,33345.416566,397


saved: artifacts\reports\final_reranker_lgbm_dl__merged_with_dl_all\final_reranker_feature_importance.parquet


## Score validation/test and save final recommendations

In [20]:
val_reranked_dir, val_predict_time_sec = score_dataset_dir_to_dir(
    best_selection_record,
    val_eval_dir,
    RERANKED_DIR / "val",
    k=max(RCFG.metric_ks),
)

# Если test labeled, скорим до max(metric_ks), чтобы корректно посчитать test@200.
# Финальный список рекомендаций ниже все равно сохраняется только top RCFG.final_k.
test_score_k = max(max(RCFG.metric_ks), RCFG.final_k) if RCFG.test_has_labels else RCFG.final_k
test_reranked_dir, test_predict_time_sec = score_dataset_dir_to_dir(
    production_record,
    test_dataset_dir,
    RERANKED_DIR / "test",
    k=test_score_k,
)

best_val_metrics = evaluate_scored_dir(
    val_reranked_dir,
    val_gt,
    rank_col="rank",
    label=f"best_{best_model_name}",
)

best_val_metrics_path = REPORTS_DIR / "final_reranker_best_val_metrics.parquet"
best_val_metrics.to_parquet(best_val_metrics_path, index=False)
display(best_val_metrics)
print("saved:", rel(best_val_metrics_path))

if RCFG.test_has_labels and test_eval_gt:
    test_metrics = evaluate_scored_dir(
        test_reranked_dir,
        test_eval_gt,
        rank_col="rank",
        label=f"test_{production_record['name']}",
    )
else:
    test_metrics = pd.DataFrame()

test_metrics_path = REPORTS_DIR / "final_reranker_test_metrics.parquet"
test_metrics.to_parquet(test_metrics_path, index=False)
display(test_metrics)
print("saved:", rel(test_metrics_path))


Score lgbm_ranker on data\processed\reranker\final_reranker_lgbm_dl__merged_with_dl_all\dataset\val_eval:   0%…

lgbm_ranker: scored rows=71,985,600, time=599.2s -> data\datamart\candidates\reranked\final_reranker_lgbm_dl__merged_with_dl_all\val


Score lgbm_ranker on data\processed\reranker\final_reranker_lgbm_dl__merged_with_dl_all\dataset\test:   0%|   …

lgbm_ranker: scored rows=69,351,800, time=580.3s -> data\datamart\candidates\reranked\final_reranker_lgbm_dl__merged_with_dl_all\test


Evaluate best_lgbm_ranker:   0%|          | 0/128 [00:00<?, ?it/s]

,model,k,users_total,users_with_predictions,user_coverage,hit_rate,recall,precision,ndcg,map,mrr
0,best_lgbm_ranker,20,359928,359928,1.0,0.073051,0.026519,0.004201,0.016838,0.009578,0.022917
1,best_lgbm_ranker,50,359928,359928,1.0,0.117815,0.042339,0.002923,0.021144,0.010151,0.024322
2,best_lgbm_ranker,100,359928,359928,1.0,0.161291,0.059340,0.002147,0.025054,0.010449,0.024938
3,best_lgbm_ranker,200,359928,359928,1.0,0.210617,0.081128,0.001501,0.029363,0.010652,0.025291


saved: artifacts\reports\final_reranker_lgbm_dl__merged_with_dl_all\final_reranker_best_val_metrics.parquet


Evaluate test_lgbm_ranker:   0%|          | 0/128 [00:00<?, ?it/s]

,model,k,users_total,users_with_predictions,user_coverage,hit_rate,recall,precision,ndcg,map,mrr
0,test_lgbm_ranker,20,346759,346759,1.0,0.061965,0.025068,0.003441,0.015201,0.008992,0.018475
1,test_lgbm_ranker,50,346759,346759,1.0,0.108260,0.041730,0.002574,0.019686,0.009575,0.019918
2,test_lgbm_ranker,100,346759,346759,1.0,0.155018,0.060276,0.001973,0.023862,0.009888,0.020582
3,test_lgbm_ranker,200,346759,346759,1.0,0.207109,0.083350,0.001411,0.028349,0.010096,0.020957


saved: artifacts\reports\final_reranker_lgbm_dl__merged_with_dl_all\final_reranker_test_metrics.parquet


In [21]:
def build_fallback_items(context_start: date, context_end: date, *, pool_size: int = RCFG.fallback_pool_size) -> list[int]:
    frames: list[pl.LazyFrame] = []

    if path_has_parquet(orders_path("item")):
        orders = read_orders_lf("item", context_start, context_end)
    else:
        orders = read_orders_lf("user_item", context_start, context_end)
    orders = ensure_numeric_cols(orders, ORDER_METRIC_COLS)
    frames.append(
        orders.group_by("item_id").agg(
            pl.sum("cnt_orders").alias("cnt_orders"),
            pl.sum("cnt_delivered_orders").alias("cnt_delivered_orders"),
            pl.sum("cnt_canceled_orders").alias("cnt_canceled_orders"),
        )
    )

    if path_has_parquet(events_path("item")):
        events = read_events_lf("item", context_start, context_end)
    elif path_has_parquet(events_path("user_item")):
        events = read_events_lf("user_item", context_start, context_end)
    else:
        events = None

    if events is not None:
        events = ensure_numeric_cols(events, EVENT_METRIC_COLS)
        frames.append(
            events.group_by("item_id").agg(
                pl.sum("cnt_events").alias("cnt_events"),
                pl.sum("cnt_to_cart").alias("cnt_to_cart"),
                pl.sum("cnt_favorite").alias("cnt_favorite"),
                pl.sum("cnt_view_description").alias("cnt_view_description"),
            )
        )

    base = frames[0]
    for part in frames[1:]:
        base = base.join(part, on="item_id", how="outer_coalesce")

    for col in ["cnt_orders", "cnt_delivered_orders", "cnt_canceled_orders", "cnt_events", "cnt_to_cart", "cnt_favorite", "cnt_view_description"]:
        if col not in lazy_columns(base):
            base = base.with_columns(pl.lit(0.0, dtype=pl.Float32).alias(col))

    fallback = collect_streaming(
        base.with_columns(
            (
                100.0 * pl.col("cnt_delivered_orders").fill_null(0)
                + 50.0 * pl.col("cnt_orders").fill_null(0)
                - 50.0 * pl.col("cnt_canceled_orders").fill_null(0)
                + 10.0 * pl.col("cnt_to_cart").fill_null(0)
                + 3.0 * pl.col("cnt_favorite").fill_null(0)
                + 1.0 * pl.col("cnt_view_description").fill_null(0)
                + 0.1 * pl.col("cnt_events").fill_null(0)
            ).cast(pl.Float32).alias("fallback_score")
        )
        .sort(["fallback_score", "item_id"], descending=[True, False])
        .select(pl.col("item_id").cast(pl.Int64))
        .head(int(pool_size))
    )
    return [int(x) for x in fallback["item_id"].to_list()]


def save_long_recommendations(scored_dir: Path, out_path: Path, *, k: int = RCFG.final_k) -> Path:
    parts = []
    for file in candidate_files(scored_dir):
        part = pl.read_parquet(file).select(
            pl.col("user_id").cast(pl.Int64),
            pl.col("item_id").cast(pl.Int64),
            pl.col("rank").cast(pl.Int32),
            pl.col("ranker_score").cast(pl.Float32),
        )
        parts.append(part)

    if not parts:
        raise RuntimeError(f"No scored files in {scored_dir}")

    out = pl.concat(parts, how="vertical_relaxed").filter(pl.col("rank") <= int(k)).sort(["user_id", "rank"])
    ensure_dir(out_path.parent)
    out.write_parquet(out_path, compression="zstd")
    print(f"saved: rows={out.height:,} -> {rel(out_path)}")
    return out_path


def save_list_recommendations(
    scored_dir: Path,
    out_path: Path,
    *,
    k: int = RCFG.final_k,
    required_user_ids: Sequence[int] | None = None,
    fill_missing: bool = RCFG.fill_missing_test_users_with_top_popular,
) -> Path:
    parts = []
    for file in candidate_files(scored_dir):
        part = pl.read_parquet(file).select(
            pl.col("user_id").cast(pl.Int64),
            pl.col("item_id").cast(pl.Int64),
            pl.col("rank").cast(pl.Int32),
        )
        parts.append(part)

    if not parts:
        raise RuntimeError(f"No scored files in {scored_dir}")

    pred = (
        pl.concat(parts, how="vertical_relaxed")
        .filter(pl.col("rank") <= int(k))
        .sort(["user_id", "rank"])
        .group_by("user_id", maintain_order=True)
        .agg(pl.col("item_id").alias("item_ids"))
        .sort("user_id")
    )

    if fill_missing and required_user_ids is not None:
        fallback_items = build_fallback_items(RCFG.full_context_start, RCFG.full_context_end, pool_size=max(RCFG.fallback_pool_size, k))
        fallback_items = fallback_items[: max(RCFG.fallback_pool_size, k)]
        pred_map = {int(u): [int(x) for x in items] for u, items in zip(pred["user_id"], pred["item_ids"])}
        rows = []
        for user_id in sorted(map(int, required_user_ids)):
            items = list(dict.fromkeys(pred_map.get(user_id, [])))
            if len(items) < k:
                seen = set(items)
                for item in fallback_items:
                    if item not in seen:
                        items.append(int(item))
                        seen.add(int(item))
                    if len(items) >= k:
                        break
            rows.append((user_id, items[:k]))
        pred = pl.DataFrame({
            "user_id": [u for u, _ in rows],
            "item_ids": [items for _, items in rows],
        }, schema={"user_id": pl.Int64, "item_ids": pl.List(pl.Int64)})

    ensure_dir(out_path.parent)
    pred.write_parquet(out_path, compression="zstd")
    print(f"saved: rows={pred.height:,} -> {rel(out_path)}")
    if required_user_ids is not None:
        print(f"required users={len(required_user_ids):,}, output users={pred.height:,}")
    return out_path


test_long_path = PROCESSED_DIR / "test_recommendations_long.parquet"
test_list_path = PROCESSED_DIR / "test_recommendations_list.parquet"

save_long_recommendations(test_reranked_dir, test_long_path, k=RCFG.final_k)
save_list_recommendations(
    test_reranked_dir,
    test_list_path,
    k=RCFG.final_k,
    required_user_ids=test_users,
    fill_missing=RCFG.fill_missing_test_users_with_top_popular,
)

saved: rows=34,675,900 -> data\processed\reranker\final_reranker_lgbm_dl__merged_with_dl_all\test_recommendations_long.parquet


C:\Users\denis\AppData\Local\Temp\ipykernel_16760\1683789037.py:37: DeprecationWarning: use of `how='outer_coalesce'` should be replaced with `how='full', coalesce=True`.
(Deprecated in version 0.20.29)
  base = base.join(part, on="item_id", how="outer_coalesce")


saved: rows=346,759 -> data\processed\reranker\final_reranker_lgbm_dl__merged_with_dl_all\test_recommendations_list.parquet
required users=346,759, output users=346,759


WindowsPath('c:/Users/denis/Documents/Repo/ozon/data/processed/reranker/final_reranker_lgbm_dl__merged_with_dl_all/test_recommendations_list.parquet')

In [22]:
final_comparison_row = model_comparison.loc[model_comparison["model"] == best_model_name].iloc[0].to_dict()
test_optimize_metric_value = (
    metric_from_metrics_df(test_metrics, RCFG.optimize_metric)
    if "test_metrics" in globals() and isinstance(test_metrics, pd.DataFrame) and not test_metrics.empty
    else None
)

final_runtime_summary = {
    "best_model": best_model_name,
    "optimize_metric": RCFG.optimize_metric,
    "best_validation_metric": float(final_comparison_row["optimize_metric_value"]),
    "test_holdout_metric": float(test_optimize_metric_value) if test_optimize_metric_value is not None else None,
    "selection_val_predict_time_sec": float(final_comparison_row["val_predict_time_sec"]),
    "production_fit_time_sec": float(production_record["production_fit_time_sec"]),
    "test_predict_time_sec": float(test_predict_time_sec),
}

manifest = {
    "built_at_utc": datetime.now(timezone.utc).isoformat(),
    "run_name": RUN_NAME,
    "active_candidate_input_name": ACTIVE_CANDIDATE_INPUT_NAME,
    "config": {k: str(v) if isinstance(v, date) else v for k, v in asdict(RCFG).items()},
    "model_selection": final_runtime_summary,
    "available_models": [r["name"] for r in model_records],
    "inputs": {
        "datamart": rel(DATAMART_DIR),
        "train_candidates": rel(train_candidates_dir),
        "val_candidates": rel(val_candidates_dir),
        "test_candidates": rel(test_candidates_dir),
    },
    "outputs": {
        "train_fit_dataset": rel(train_fit_dir),
        "val_fit_dataset": rel(val_fit_dir),
        "val_eval_dataset": rel(val_eval_dir),
        "test_dataset": rel(test_dataset_dir),
        "production_model_paths": production_model_paths,
        "features": rel(feature_list_path),
        "val_reranked": rel(val_reranked_dir),
        "test_reranked": rel(test_reranked_dir),
        "test_recommendations_long": rel(test_long_path),
        "test_recommendations_list": rel(test_list_path),
        "baseline_val_metrics": rel(baseline_metrics_path),
        "best_val_metrics": rel(best_val_metrics_path),
        "test_metrics": rel(test_metrics_path),
        "model_comparison": rel(model_comparison_path),
        "model_val_metrics": rel(model_val_metrics_path),
        "feature_importance": rel(importance_path),
        "dataset_overview": rel(overview_path),
    },
    "quality_notes": [
        "Default input is merged_with_dl_all: base non-DL + DSSM + BERT4Rec + SASRec selected quotas.",
        "Features are point-in-time and use only context windows before target windows.",
        "Candidate-scoped user_item features are separated by active candidate pool to avoid stale base-reranker cache reuse.",
        "LightGBM LambdaRank is default production model; CatBoost is optional challenger.",
        "Fit datasets keep all positives and downsample negatives per user at build time.",
        "Model matrices are additionally memory-capped: limited feature count and sampled negative rows before pandas conversion.",
        "Previous ArrowMemoryError was caused by full train_fit 19.3M x 566 pandas conversion; this notebook avoids that.",
        "Test is treated as a labeled holdout when RCFG.test_has_labels=True; use it once for final reporting, not for model selection.",
        "Final test list can fill missing test users with global TopPopular fallback.",
    ],
}

manifest_path = PROCESSED_DIR / "manifest_final_reranker.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved:", rel(manifest_path))

print("Model comparison:")
display(model_comparison)
print("Baseline validation metrics:")
display(baseline_metrics.sort_values(["model", "k"]) if not baseline_metrics.empty else baseline_metrics)
print("Best model validation metrics:")
display(best_val_metrics)
print("Test holdout metrics:")
display(test_metrics)
print("Top feature importances:")
display(importance.head(30))
print("Final runtime summary:")
print(json.dumps(final_runtime_summary, ensure_ascii=False, indent=2))


saved: data\processed\reranker\final_reranker_lgbm_dl__merged_with_dl_all\manifest_final_reranker.json
Model comparison:


,model,model_type,features_count,categorical_features,tuning_time_sec,selection_fit_time_sec,val_predict_time_sec,best_iteration,model_paths,ndcg@20,map@20,recall@20,hit_rate@20,ndcg@50,map@50,recall@50,hit_rate@50,ndcg@100,map@100,recall@100,hit_rate@100,ndcg@200,map@200,recall@200,hit_rate@200,optimize_metric_value
0,lgbm_ranker,lgbm,180,,956.912602,10.6015,606.906563,53,"{""joblib"": ""models\\reranker\\final_reranker_l...",0.016838,0.009578,0.026519,0.073051,0.021144,0.010151,0.042339,0.117815,0.025054,0.010449,0.05934,0.161291,0.029363,0.010652,0.081128,0.210617,0.025054


Baseline validation metrics:


,model,k,users_total,users_with_predictions,user_coverage,hit_rate,recall,precision,ndcg,map,mrr
8,als_rank,20,359928,359928,1.0,0.005759,0.001496,0.000292,0.000750,0.000271,0.001010
9,als_rank,50,359928,359928,1.0,0.014067,0.003845,0.000290,0.001374,0.000342,0.001261
10,als_rank,100,359928,359928,1.0,0.027644,0.007678,0.000292,0.002240,0.000397,0.001449
11,als_rank,200,359928,359928,1.0,0.053413,0.015607,0.000295,0.003774,0.000455,0.001627
0,candidate_rank,20,359928,359928,1.0,0.030059,0.012853,0.001608,0.006312,0.003013,0.006905
1,candidate_rank,50,359928,359928,1.0,0.061157,0.023976,0.001372,0.009198,0.003376,0.007839
2,candidate_rank,100,359928,359928,1.0,0.110186,0.042449,0.001315,0.013222,0.003664,0.008524
3,candidate_rank,200,359928,359928,1.0,0.164541,0.064736,0.001056,0.017515,0.003857,0.008917
24,dl_bert4rec_small_rank,20,359928,359928,1.0,0.007921,0.002178,0.000401,0.001060,0.000380,0.001399
25,dl_bert4rec_small_rank,50,359928,359928,1.0,0.019573,0.005762,0.000404,0.001992,0.000489,0.001749


Best model validation metrics:


,model,k,users_total,users_with_predictions,user_coverage,hit_rate,recall,precision,ndcg,map,mrr
0,best_lgbm_ranker,20,359928,359928,1.0,0.073051,0.026519,0.004201,0.016838,0.009578,0.022917
1,best_lgbm_ranker,50,359928,359928,1.0,0.117815,0.042339,0.002923,0.021144,0.010151,0.024322
2,best_lgbm_ranker,100,359928,359928,1.0,0.161291,0.059340,0.002147,0.025054,0.010449,0.024938
3,best_lgbm_ranker,200,359928,359928,1.0,0.210617,0.081128,0.001501,0.029363,0.010652,0.025291


Test holdout metrics:


,model,k,users_total,users_with_predictions,user_coverage,hit_rate,recall,precision,ndcg,map,mrr
0,test_lgbm_ranker,20,346759,346759,1.0,0.061965,0.025068,0.003441,0.015201,0.008992,0.018475
1,test_lgbm_ranker,50,346759,346759,1.0,0.108260,0.041730,0.002574,0.019686,0.009575,0.019918
2,test_lgbm_ranker,100,346759,346759,1.0,0.155018,0.060276,0.001973,0.023862,0.009888,0.020582
3,test_lgbm_ranker,200,346759,346759,1.0,0.207109,0.083350,0.001411,0.028349,0.010096,0.020957


Top feature importances:


,feature,importance_gain,importance_split
54,f_num__candidate__i2i_bm25_score,323799.488817,339
6,f_num__candidate__candidate_rank,130356.412301,571
15,f_num__candidate__clip_hnsw_rank,117822.750261,228
8,f_num__candidate__candidate_rank_rr,107908.191088,311
55,f_num__candidate__i2i_bm25_score_log1p_nonnega...,107461.280681,180
2,f_num__candidate__als_rank_rr,97134.331371,109
53,f_num__candidate__i2i_bm25_rank_rr,75393.340089,285
30,f_num__candidate__dl_dssm_final_pop_prior,72788.696748,227
0,f_num__candidate__als_rank,69399.703131,270
3,f_num__candidate__als_score,33345.416566,397


Final runtime summary:
{
  "best_model": "lgbm_ranker",
  "optimize_metric": "ndcg@100",
  "best_validation_metric": 0.02505409551450728,
  "test_holdout_metric": 0.023861575329100267,
  "selection_val_predict_time_sec": 606.9065633999999,
  "production_fit_time_sec": 0.0,
  "test_predict_time_sec": 580.3287959000008
}
